# Modelado Predictivo de la Infracción D04

Se desarrolla un modelo predictivo para la infracción D04 (no detenerse ante una luz roja o amarilla de semáforo, una señal de «pare» o un semáforo intermitente en rojo) empleando una metodología iterativa en la que cada nuevo modelo aprende de los resultados obtenidos por los modelos previamente entrenados. El proceso inicia con el modelo Holt-Winters, cuyos parámetros se ajustan con base en los hallazgos del análisis exploratorio y la descomposición STL de la serie temporal. Posteriormente, se incorporan secuencialmente los modelos ARIMA, SARIMA, Dynamic Optimized Theta, Ridge, Lasso, Random Forest, XGBoost, LightGBM y KNN, de modo que cada uno aprovecha el conocimiento acumulado por sus predecesores, con el objetivo de mejorar progresivamente el rendimiento predictivo. Las métricas de evaluación utilizadas son RMSE, MAE, MAPE, SMAPE y MSE. Finalmente, se selecciona el mejor modelo para realizar pronósticos del volumen de infracciones por paso de semáforo en rojo para el año 2026, cuyos datos aún no han sido publicados por la Alcaldía de Barranquilla a través de su portal de Datos Abiertos.

## Carga de librerías

Se importan las librerías necesarias para el desarrollo de modelos predictivos sobre las series temporales de comparendos electrónicos. Se emplean `pandas` y `numpy` para la manipulación y transformación de datos, `plotly.graph_objects` y `plotly.subplots` para la visualización de resultados, y `statsmodels.tsa` para los modelos estadísticos clásicos (Holt-Winters, ARIMA, SARIMA y STL). Para los enfoques de machine learning, se utilizan `scikit-learn` con `RidgeCV`, `MultiTaskLassoCV`, `RandomForestRegressor`, `KNeighborsRegressor` y `GridSearchCV` para optimización de hiperparámetros, junto con `xgboost` (XGBoost) y `lightgbm` (LightGBM). Adicionalmente, se incorporan `sklearn.model_selection.TimeSeriesSplit` para validación con series temporales, `sklearn.preprocessing.StandardScaler` para estandarización de características, y `sklearn.metrics` para las métricas de evaluación (RMSE, MAE, MAPE, SMAPE y MSE). Se suprimen los mensajes de advertencia para mantener una salida limpia y enfocada en los resultados.

In [1]:
import warnings

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import STL
from scipy import stats as sp_stats
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import RidgeCV
from sklearn.linear_model import MultiTaskLassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.neighbors import KNeighborsRegressor

warnings.filterwarnings('ignore')

In [2]:
pio.renderers.default = "notebook_connected"

## Carga del DataFrame

Se carga el archivo CSV que contiene los datos de comparendos electrónicos utilizando la función `read_csv` de la librería pandas, y se almacena en el DataFrame `df_comparendos_electronicos`.

In [3]:
df_comparendos_electronicos = pd.read_csv("C:/Users/david/Documents/seminario_investigativo/comparendos_electronicos.csv")

### Conversión de las Fechas a Formato datetime

Se convierte la columna `fecha_comparendo` al tipo `datetime` utilizando el formato especificado (`'%Y %b %d %I:%M:%S %p'`), normalizando la hora a medianoche con el método `.dt.normalize()` para eliminar la información horaria y trabajar únicamente con fechas, dado que todos los registros contienen la hora de 12:00:00 AM. Posteriormente, se imprime el tipo de dato resultante para verificar la correcta conversión.

In [4]:
df_comparendos_electronicos['fecha_comparendo'] = pd.to_datetime(df_comparendos_electronicos['fecha_comparendo'], format='%Y %b %d %I:%M:%S %p').dt.normalize()

print(df_comparendos_electronicos['fecha_comparendo'].dtype)

datetime64[ns]


### Corrección de Valores Nulos

Se rellenan los valores nulos de la columna `DESC_INFRACCION` con la descripción correspondiente al código C14, obtenida del sitio web oficial del Tránsito del Atlántico. Según esta fuente, la infracción C14 corresponde a **"Transitar por sitios restringidos o en horas prohibidas por la autoridad competente"**. Esta corrección se aplica exclusivamente a los 564 registros que presentaban valores nulos, asociados a las nuevas cámaras tipo Carril Bus implementadas en Barranquilla a partir del 17 de octubre de 2025.

**Fuente:** https://transitodelatlantico.gov.co/valor-de-multas-de-transito/

In [5]:
df_comparendos_electronicos['DESC_INFRACCION'] = df_comparendos_electronicos['DESC_INFRACCION'].fillna("Transitar por sitios restringidos o en horas prohibidas por la autoridad competente")  

print(f"Total de valores nulos en el DataFrame: {df_comparendos_electronicos.isnull().sum().sum()}")

Total de valores nulos en el DataFrame: 0


### Corrección de Cámaras Duplicadas

Se unifican los nombres de las cámaras que presentan dos notaciones diferentes para el mismo punto de control. Esta corrección permite evitar la duplicación de ubicaciones en los análisis y garantizar la consistencia de los datos.

In [6]:
df_comparendos_electronicos.loc[df_comparendos_electronicos['Camara_y_direccion'] == 'CARRERA 53 CON CALLE 104', 'Camara_y_direccion'] = 'CARRERA 53 ENTRE CALLE 104 Y 106'

df_comparendos_electronicos.loc[df_comparendos_electronicos['Camara_y_direccion'] == 'CALLE 84 CON CARRERA 59', 'Camara_y_direccion'] = 'CALLE 84 ENTRE CARRERA 59 Y 59B'

df_comparendos_electronicos.loc[df_comparendos_electronicos['Camara_y_direccion'] == 'CARRERA 45 CON CALLE 53', 'Camara_y_direccion'] = 'CALLE 53 CON CARRERA 45'

df_comparendos_electronicos.loc[df_comparendos_electronicos['Camara_y_direccion'] == 'CALLE 45B CARRERA 14', 'Camara_y_direccion'] = 'CALLE 45B CON CARRERA 14'

### Corrección de Infracciones C02 Detectadas por Cámaras Fijas

Dado que estos registros representan una inconsistencia en la base de datos (las cámaras fijas no están diseñadas para detectar estacionamiento prohibido), se procede a eliminarlos del DataFrame principal para garantizar la consistencia de los análisis posteriores. Posteriormente, se verifica que no queden registros residuales de C02 asociados a cámaras fijas, confirmando la correcta limpieza de los datos.

In [7]:
df_comparendos_electronicos = df_comparendos_electronicos[~((df_comparendos_electronicos['COD_INFRACCION'] == 'C02') & 
                                                             (df_comparendos_electronicos['Tipo Camara'] == 'Fijo'))]

## Preparación de la Serie Temporal Mensual

Se extrae y configura la serie temporal mensual correspondiente a la infracción D04 (no detenerse ante una luz roja o amarilla de semáforo, una señal de «pare» o un semáforo intermitente en rojo) a partir del DataFrame de comparendos electrónicos. Para ello, se agrupan los registros por mes y código de infracción, sumando la columna `CANTIDAD_INFRACCIONES` para obtener el volumen real de infracciones por período. La serie resultante se indexa con fechas mensuales, se establece una frecuencia mensual ('MS') y se rellenan los valores faltantes con cero, asegurando una secuencia temporal continua y completa para los modelos predictivos. Se imprime información básica de la serie: número de meses, rango temporal y total de comparendos del período.

In [8]:
df_comparendos_electronicos_copy = df_comparendos_electronicos.copy()
df_comparendos_electronicos_copy['año_mes'] = df_comparendos_electronicos_copy['fecha_comparendo'].dt.to_period('M').astype(str)

def preparar_serie_mensual(df, codigo_infraccion):
    infracciones_por_mes = df.groupby(['año_mes', 'COD_INFRACCION'])['CANTIDAD_INFRACCIONES'].sum().reset_index()
    
    serie = infracciones_por_mes[infracciones_por_mes['COD_INFRACCION'] == codigo_infraccion].copy()
    serie = serie.set_index('año_mes')['CANTIDAD_INFRACCIONES']
    
    serie.index = pd.to_datetime(serie.index)
    serie = serie.asfreq('MS')
    serie = serie.fillna(0)
    
    return serie

serie_d04 = preparar_serie_mensual(df_comparendos_electronicos_copy, 'D04')
print(f"Serie D04: {len(serie_d04)} meses")
print(f"Desde: {serie_d04.index.min().strftime('%Y-%m')} hasta: {serie_d04.index.max().strftime('%Y-%m')}")
print(f"Total comparendos: {serie_d04.sum():,.0f}")

Serie D04: 96 meses
Desde: 2018-01 hasta: 2025-12
Total comparendos: 54,303


## División de la Serie Temporal en Conjuntos de Entrenamiento y Prueba

Se particiona la serie temporal mensual de la infracción D04 (no detenerse ante una luz roja o amarilla de semáforo, una señal de «pare» o un semáforo intermitente en rojo) en dos conjuntos: entrenamiento y prueba. El conjunto de entrenamiento comprende los meses desde enero de 2018 hasta diciembre de 2024, abarcando 84 meses, mientras que el conjunto de prueba corresponde al año completo de 2025, con 12 meses. Esta división permite entrenar los modelos predictivos con los datos históricos y evaluar su desempeño pronosticando el período más reciente (2025), cuyos valores reales ya están disponibles en la base de datos para calcular las métricas de error. Se imprime información detallada de ambos conjuntos, incluyendo su rango temporal y número de meses.

In [9]:
train_start = '2018-01-01'
train_end = '2024-12-01'
test_start = '2025-01-01'
test_end = '2025-12-01'

train_d04 = serie_d04[train_start:train_end].copy()
test_d04 = serie_d04[test_start:test_end].copy()

print(f"Train: {train_d04.index.min().strftime('%Y-%m')} a {train_d04.index.max().strftime('%Y-%m')} ({len(train_d04)} meses)")
print(f"Test: {test_d04.index.min().strftime('%Y-%m')} a {test_d04.index.max().strftime('%Y-%m')} ({len(test_d04)} meses)")

Train: 2018-01 a 2024-12 (84 meses)
Test: 2025-01 a 2025-12 (12 meses)


## Generación de Ventanas de Validación Cruzada Temporal

Se implementa una función de validación cruzada específica para series temporales, que respeta el orden cronológico de los datos y evita la fuga de información del futuro. La función `time_series_cv_manual` genera un conjunto de ventanas de entrenamiento y validación, donde el tamaño de cada ventana de validación es de 12 meses (un año completo). Se configuran 4 ventanas de validación, lo que significa que el modelo se evalúa con los últimos 4 años de datos disponibles. Para la infracción D04, se imprimen los rangos temporales de cada fold, mostrando el período de entrenamiento y validación correspondiente. Esta metodología permite evaluar el desempeño predictivo de los modelos en diferentes horizontes temporales de manera robusta y realista.

In [10]:
def time_series_cv_manual(serie, n_ventanas=4, horizonte=12):
    ventanas = []
    
    for i in range(n_ventanas, 0, -1):
        train_end_idx = len(serie) - (i * horizonte)
        train_fold = serie.iloc[:train_end_idx]
        val_fold = serie.iloc[train_end_idx:train_end_idx + horizonte]
        
        ventanas.append((train_fold, val_fold))
        
    return ventanas

ventanas_d04 = time_series_cv_manual(train_d04, n_ventanas=4, horizonte=12)

print("Ventanas de validación generadas:")
for i, (train_fold, val_fold) in enumerate(ventanas_d04, 1):
    print(f"Fold {i}: Train {train_fold.index.min().strftime('%Y-%m')} a {train_fold.index.max().strftime('%Y-%m')} "
          f"({len(train_fold)}m) -> Val {val_fold.index.min().strftime('%Y-%m')} a {val_fold.index.max().strftime('%Y-%m')} "
          f"({len(val_fold)}m)")

Ventanas de validación generadas:
Fold 1: Train 2018-01 a 2020-12 (36m) -> Val 2021-01 a 2021-12 (12m)
Fold 2: Train 2018-01 a 2021-12 (48m) -> Val 2022-01 a 2022-12 (12m)
Fold 3: Train 2018-01 a 2022-12 (60m) -> Val 2023-01 a 2023-12 (12m)
Fold 4: Train 2018-01 a 2023-12 (72m) -> Val 2024-01 a 2024-12 (12m)


## Holt-Winters

In [11]:
def evaluar_holt_winters(train_fold, val_fold, trend=None, seasonal=None,
                         damped_trend=False, seasonal_periods=None):
    """
    Ajusta un modelo de suavizamiento exponencial (Holt-Winters)
    con los componentes especificados.
    """
    modelo = ExponentialSmoothing(
        train_fold,
        trend=trend,
        seasonal=seasonal,
        damped_trend=damped_trend,
        seasonal_periods=seasonal_periods,
        initialization_method='estimated'
    ).fit()
    
    predicciones = modelo.forecast(len(val_fold))
    
    real = val_fold.values
    pred = predicciones.values
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, modelo

In [12]:
configuraciones_hw = [
    {'trend': None, 'seasonal': None, 'damped': False, 'nombre': 'SES (Simple)'},
    {'trend': 'add', 'seasonal': None, 'damped': False, 'nombre': 'Holt lineal'},
    {'trend': 'add', 'seasonal': None, 'damped': True,  'nombre': 'Holt amortiguado'},
    {'trend': None, 'seasonal': 'add', 'damped': False, 'nombre': 'Estacional aditiva'},
    {'trend': 'add', 'seasonal': 'add', 'damped': False, 'nombre': 'Holt‑Winters aditivo'},
    {'trend': 'add', 'seasonal': 'add', 'damped': True,  'nombre': 'HW aditivo amortiguado'},
]

resultados_cv_d04 = []

print("-" * 50)
print("Validación cruzada - Holt-Winters (D04)")
print("-" * 50)

for config in configuraciones_hw:
    nombre = config['nombre']
    print(f"\n--- {nombre} ---")
    
    metricas_folds = {'RMSE': [], 'MAE': [], 'MAPE': [], 'SMAPE': [], 'MSE': []}
    
    for i, (train_fold, val_fold) in enumerate(ventanas_d04, 1):
        pred, met, _ = evaluar_holt_winters(
            train_fold, val_fold,
            trend=config['trend'],
            seasonal=config['seasonal'],
            damped_trend=config['damped'],
            seasonal_periods=12 if config['seasonal'] else None
        )
        metricas_folds['RMSE'].append(met['RMSE'])
        metricas_folds['MAE'].append(met['MAE'])
        metricas_folds['MAPE'].append(met['MAPE'])
        metricas_folds['SMAPE'].append(met['SMAPE'])
        metricas_folds['MSE'].append(met['MSE'])
        
        print(f"  Fold {i}: RMSE={met['RMSE']:.2f}, MAE={met['MAE']:.2f}, "
              f"MAPE={met['MAPE']:.2f}%, SMAPE={met['SMAPE']:.2f}%, MSE={met['MSE']:.2f}")
    
    promedio = {k: np.mean(v) for k, v in metricas_folds.items()}
    resultados_cv_d04.append({
        'modelo': nombre,
        'RMSE': promedio['RMSE'],
        'MAE': promedio['MAE'],
        'MAPE': promedio['MAPE'],
        'SMAPE': promedio['SMAPE'],
        'MSE': promedio['MSE']
    })
    
    print(f"  Promedio -> RMSE={promedio['RMSE']:.2f}, MAE={promedio['MAE']:.2f}, "
          f"MAPE={promedio['MAPE']:.2f}%, SMAPE={promedio['SMAPE']:.2f}%, MSE={promedio['MSE']:.2f}")

df_cv_d04 = pd.DataFrame(resultados_cv_d04)
mejor_idx = df_cv_d04['RMSE'].idxmin()
mejor_nombre = df_cv_d04.loc[mejor_idx, 'modelo']
mejor_config = next(c for c in configuraciones_hw if c['nombre'] == mejor_nombre)

print("\n" + "-" * 50)
print("Mejor modelo Holt‑Winters según RMSE en validación cruzada")
print("-" * 50)
print(f"Modelo seleccionado: {mejor_nombre}")
display(df_cv_d04.round(2).sort_values('RMSE'))

--------------------------------------------------
Validación cruzada - Holt-Winters (D04)
--------------------------------------------------

--- SES (Simple) ---
  Fold 1: RMSE=128.54, MAE=107.46, MAPE=36.24%, SMAPE=27.49%, MSE=16521.27
  Fold 2: RMSE=368.46, MAE=308.37, MAPE=36.96%, SMAPE=47.73%, MSE=135764.00
  Fold 3: RMSE=363.65, MAE=304.67, MAPE=78.95%, SMAPE=48.13%, MSE=132241.64
  Fold 4: RMSE=281.86, MAE=229.86, MAPE=29.77%, SMAPE=36.47%, MSE=79444.01
  Promedio -> RMSE=285.63, MAE=237.59, MAPE=45.48%, SMAPE=39.96%, MSE=90992.73

--- Holt lineal ---
  Fold 1: RMSE=123.90, MAE=104.69, MAPE=34.84%, SMAPE=26.95%, MSE=15350.29
  Fold 2: RMSE=377.52, MAE=317.59, MAPE=38.16%, SMAPE=49.66%, MSE=142520.40
  Fold 3: RMSE=405.85, MAE=342.98, MAPE=88.13%, SMAPE=51.91%, MSE=164713.35
  Fold 4: RMSE=283.60, MAE=231.30, MAPE=29.93%, SMAPE=36.76%, MSE=80428.72
  Promedio -> RMSE=297.72, MAE=249.14, MAPE=47.76%, SMAPE=41.32%, MSE=100753.19

--- Holt amortiguado ---
  Fold 1: RMSE=131.10, MAE

,modelo,RMSE,MAE,MAPE,SMAPE,MSE
0,SES (Simple),285.63,237.59,45.48,39.96,90992.73
3,Estacional aditiva,293.30,241.41,43.34,42.36,97337.58
2,Holt amortiguado,294.59,245.54,47.46,40.79,97446.47
1,Holt lineal,297.72,249.14,47.76,41.32,100753.19
5,HW aditivo amortiguado,307.39,258.10,46.42,46.36,104142.57
4,Holt‑Winters aditivo,309.62,257.64,46.41,45.10,108958.37


In [13]:
print("-" * 50)
print(f"Evaluación final en Test (2025) - {mejor_nombre}")
print("-" * 50)

modelo_final_d04 = ExponentialSmoothing(
    train_d04,
    trend=mejor_config['trend'],
    seasonal=mejor_config['seasonal'],
    damped_trend=mejor_config['damped'],
    seasonal_periods=12 if mejor_config['seasonal'] else None,
    initialization_method='estimated'
).fit()

pred_test_d04 = modelo_final_d04.forecast(len(test_d04))

real_test = test_d04.values
pred_test = pred_test_d04.values

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = real_test - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - SES (Simple)
--------------------------------------------------
RMSE: 250.61
MAE: 235.12
MAPE: 111.22%
SMAPE: 58.52%
MSE: 62805.11


In [14]:
meses = ['Enero','Febrero','Marzo','Abril','Mayo','Junio','Julio','Agosto','Septiembre','Octubre','Noviembre','Diciembre']

fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_d04.index, y=train_d04.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_d04.index, y=test_d04.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_d04.index, y=pred_test_d04.values,
    mode='lines+markers', name=f'Predicción {mejor_nombre}',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'D04 - {mejor_nombre}: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_d04.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_d04.values,
    mode='lines+markers', name=f'Predicción {mejor_nombre}',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_d04.values - pred_test_d04.values

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## ARIMA

In [15]:
def evaluar_arima(train_fold, val_fold, order=(1,0,0)):
    """
    Ajusta un modelo ARIMA con el orden (p,d,q) dado.
    Retorna predicciones, métricas y el modelo ajustado.
    """
    modelo = ARIMA(train_fold, order=order)
    modelo_ajustado = modelo.fit()
    
    predicciones = modelo_ajustado.forecast(steps=len(val_fold))
    
    real = val_fold.values
    pred = predicciones.values
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, modelo_ajustado

In [16]:
ordenes_arima = [
    ((1,0,0), 'ARIMA(1,0,0)'),
    ((0,0,1), 'ARIMA(0,0,1)'),
    ((1,0,1), 'ARIMA(1,0,1)'),
    ((2,0,0), 'ARIMA(2,0,0)'),
    ((0,0,2), 'ARIMA(0,0,2)'),
    ((2,0,1), 'ARIMA(2,0,1)'),
    ((1,0,2), 'ARIMA(1,0,2)'),
    ((1,1,0), 'ARIMA(1,1,0)'),
    ((0,1,1), 'ARIMA(0,1,1)'),
    ((1,1,1), 'ARIMA(1,1,1)'),
]

resultados_cv_arima = []

print("-" * 50)
print("Validación cruzada - Comparación de modelos ARIMA (D04)")
print("-" * 50)

for orden, nombre in ordenes_arima:
    print(f"\n--- {nombre} ---")
    metricas_folds = {'RMSE': [], 'MAE': [], 'MAPE': [], 'SMAPE': [], 'MSE': []}
    
    for i, (train_fold, val_fold) in enumerate(ventanas_d04, 1):
        predicciones, met, _ = evaluar_arima(train_fold, val_fold, order=orden)
        metricas_folds['RMSE'].append(met['RMSE'])
        metricas_folds['MAE'].append(met['MAE'])
        metricas_folds['MAPE'].append(met['MAPE'])
        metricas_folds['SMAPE'].append(met['SMAPE'])
        metricas_folds['MSE'].append(met['MSE'])
        
        print(f"  Fold {i}: RMSE={met['RMSE']:.2f}, MAE={met['MAE']:.2f}, "
              f"MAPE={met['MAPE']:.2f}%, SMAPE={met['SMAPE']:.2f}%, MSE={met['MSE']:.2f}")
    
    promedio = {k: np.mean(v) for k, v in metricas_folds.items()}
    resultados_cv_arima.append({
        'modelo': nombre,
        'RMSE': promedio['RMSE'],
        'MAE': promedio['MAE'],
        'MAPE': promedio['MAPE'],
        'SMAPE': promedio['SMAPE'],
        'MSE': promedio['MSE']
    })
    
    print(f"  Promedio -> RMSE={promedio['RMSE']:.2f}, MAE={promedio['MAE']:.2f}, "
          f"MAPE={promedio['MAPE']:.2f}%, SMAPE={promedio['SMAPE']:.2f}%, MSE={promedio['MSE']:.2f}")

df_cv_arima = pd.DataFrame(resultados_cv_arima)
mejor_idx = df_cv_arima['RMSE'].idxmin()
mejor_orden, mejor_nombre = ordenes_arima[mejor_idx]

print("\n" + "-" * 50)
print("Mejor modelo ARIMA según RMSE en validación cruzada")
print("-" * 50)
print(f"Modelo seleccionado: {mejor_nombre}")
display(df_cv_arima.round(2).sort_values('RMSE'))

--------------------------------------------------
Validación cruzada - Comparación de modelos ARIMA (D04)
--------------------------------------------------

--- ARIMA(1,0,0) ---
  Fold 1: RMSE=223.55, MAE=192.87, MAPE=64.20%, SMAPE=42.30%, MSE=49972.72
  Fold 2: RMSE=281.28, MAE=235.68, MAPE=29.98%, SMAPE=34.13%, MSE=79115.75
  Fold 3: RMSE=222.83, MAE=184.81, MAPE=44.90%, SMAPE=33.82%, MSE=49653.98
  Fold 4: RMSE=211.70, MAE=186.75, MAPE=28.02%, SMAPE=28.83%, MSE=44816.84
  Promedio -> RMSE=234.84, MAE=200.03, MAPE=41.78%, SMAPE=34.77%, MSE=55889.82

--- ARIMA(0,0,1) ---
  Fold 1: RMSE=246.29, MAE=221.02, MAPE=71.77%, SMAPE=46.83%, MSE=60660.75
  Fold 2: RMSE=279.76, MAE=231.35, MAPE=29.50%, SMAPE=33.38%, MSE=78266.06
  Fold 3: RMSE=217.33, MAE=194.55, MAPE=44.14%, SMAPE=35.65%, MSE=47232.65
  Fold 4: RMSE=211.70, MAE=187.35, MAPE=28.12%, SMAPE=28.92%, MSE=44817.26
  Promedio -> RMSE=238.77, MAE=208.57, MAPE=43.38%, SMAPE=36.20%, MSE=57744.18

--- ARIMA(1,0,1) ---
  Fold 1: RMSE=197

,modelo,RMSE,MAE,MAPE,SMAPE,MSE
3,"ARIMA(2,0,0)",232.13,196.18,40.09,34.18,55255.02
5,"ARIMA(2,0,1)",232.67,197.89,39.92,34.47,55327.69
0,"ARIMA(1,0,0)",234.84,200.03,41.78,34.77,55889.82
2,"ARIMA(1,0,1)",235.25,198.17,40.27,34.50,57060.69
1,"ARIMA(0,0,1)",238.77,208.57,43.38,36.20,57744.18
6,"ARIMA(1,0,2)",239.17,201.95,41.62,35.10,58341.12
4,"ARIMA(0,0,2)",241.33,210.61,43.95,36.47,58944.40
9,"ARIMA(1,1,1)",259.52,218.93,43.09,37.16,76517.92
7,"ARIMA(1,1,0)",274.97,231.99,45.46,38.71,85465.61
8,"ARIMA(0,1,1)",285.26,237.31,45.46,39.88,90784.55


In [17]:
print("-" * 50)
print(f"Evaluación final en Test (2025) - {mejor_nombre}")
print("-" * 50)

modelo_final_arima = ARIMA(train_d04, order=mejor_orden)
modelo_final_arima_ajustado = modelo_final_arima.fit()

pred_test_arima = modelo_final_arima_ajustado.forecast(steps=len(test_d04))

real_test = test_d04.values
pred_test = pred_test_arima.values

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = real_test - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - ARIMA(2,0,0)
--------------------------------------------------
RMSE: 295.74
MAE: 281.03
MAPE: 130.28%
SMAPE: 65.49%
MSE: 87463.66


In [18]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_d04.index, y=train_d04.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_d04.index, y=test_d04.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_d04.index, y=pred_test_arima.values,
    mode='lines+markers', name=f'Predicción {mejor_nombre}',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'D04 - {mejor_nombre}: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_d04.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_arima.values,
    mode='lines+markers', name=f'Predicción {mejor_nombre}',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_d04.values - pred_test_arima.values

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## SARIMA

In [19]:
def evaluar_sarima(train_fold, val_fold, order=(1,0,0), seasonal_order=(0,0,0,12)):
    """
    Ajusta un modelo SARIMAX (SARIMA) con los órdenes especificados.
    Retorna predicciones, métricas y el modelo ajustado.
    """
    modelo = SARIMAX(
        train_fold,
        order=order,
        seasonal_order=seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    modelo_ajustado = modelo.fit(disp=False)
    
    predicciones = modelo_ajustado.forecast(steps=len(val_fold))
    
    real = val_fold.values
    pred = predicciones.values
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, modelo_ajustado

In [20]:
configuraciones_sarima = [
    ((1,0,0), (0,0,0,12), 'SARIMA(1,0,0)(0,0,0)[12]'),
    ((2,0,0), (0,0,0,12), 'SARIMA(2,0,0)(0,0,0)[12]'),
    ((2,0,0), (1,0,0,12), 'SARIMA(2,0,0)(1,0,0)[12]'),
    ((2,0,0), (0,0,1,12), 'SARIMA(2,0,0)(0,0,1)[12]'),
    ((2,0,0), (1,0,1,12), 'SARIMA(2,0,0)(1,0,1)[12]'),
    ((1,0,1), (1,0,0,12), 'SARIMA(1,0,1)(1,0,0)[12]'),
    ((0,0,1), (1,0,0,12), 'SARIMA(0,0,1)(1,0,0)[12]'),
    ((2,0,0), (0,1,1,12), 'SARIMA(2,0,0)(0,1,1)[12]'),
    ((1,0,0), (0,1,1,12), 'SARIMA(1,0,0)(0,1,1)[12]'),
]

resultados_cv_sarima = []

print("-" * 50)
print("Validación cruzada - Comparación de modelos SARIMA (D04)")
print("-" * 50)

for order, seasonal_order, nombre in configuraciones_sarima:
    print(f"\n--- {nombre} ---")
    metricas_folds = {'RMSE': [], 'MAE': [], 'MAPE': [], 'SMAPE': [], 'MSE': []}
    
    for i, (train_fold, val_fold) in enumerate(ventanas_d04, 1):
        _, met, _ = evaluar_sarima(train_fold, val_fold, order=order, seasonal_order=seasonal_order)
        metricas_folds['RMSE'].append(met['RMSE'])
        metricas_folds['MAE'].append(met['MAE'])
        metricas_folds['MAPE'].append(met['MAPE'])
        metricas_folds['SMAPE'].append(met['SMAPE'])
        metricas_folds['MSE'].append(met['MSE'])
        
        print(f"  Fold {i}: RMSE={met['RMSE']:.2f}, MAE={met['MAE']:.2f}, "
              f"MAPE={met['MAPE']:.2f}%, SMAPE={met['SMAPE']:.2f}%, MSE={met['MSE']:.2f}")
    
    promedio = {k: np.mean(v) for k, v in metricas_folds.items()}
    resultados_cv_sarima.append({
        'modelo': nombre,
        'RMSE': promedio['RMSE'],
        'MAE': promedio['MAE'],
        'MAPE': promedio['MAPE'],
        'SMAPE': promedio['SMAPE'],
        'MSE': promedio['MSE']
    })
    
    print(f"  Promedio -> RMSE={promedio['RMSE']:.2f}, MAE={promedio['MAE']:.2f}, "
          f"MAPE={promedio['MAPE']:.2f}%, SMAPE={promedio['SMAPE']:.2f}%, MSE={promedio['MSE']:.2f}")

df_cv_sarima = pd.DataFrame(resultados_cv_sarima)
mejor_idx = df_cv_sarima['RMSE'].idxmin()
mejor_nombre = df_cv_sarima.loc[mejor_idx, 'modelo']
mejor_order, mejor_seasonal, _ = next((o, s, n) for o, s, n in configuraciones_sarima if n == mejor_nombre)

print("\n" + "-" * 50)
print("Mejor modelo SARIMA según RMSE en validación cruzada")
print("-" * 50)
print(f"Modelo seleccionado: {mejor_nombre}")
display(df_cv_sarima.round(2).sort_values('RMSE'))

--------------------------------------------------
Validación cruzada - Comparación de modelos SARIMA (D04)
--------------------------------------------------

--- SARIMA(1,0,0)(0,0,0)[12] ---
  Fold 1: RMSE=128.42, MAE=101.74, MAPE=26.60%, SMAPE=27.16%, MSE=16492.85
  Fold 2: RMSE=412.15, MAE=350.50, MAPE=42.89%, SMAPE=57.22%, MSE=169869.25
  Fold 3: RMSE=221.89, MAE=179.62, MAPE=45.57%, SMAPE=32.94%, MSE=49233.82
  Fold 4: RMSE=318.51, MAE=269.86, MAPE=36.54%, SMAPE=45.96%, MSE=101447.13
  Promedio -> RMSE=270.24, MAE=225.43, MAPE=37.90%, SMAPE=40.82%, MSE=84260.76

--- SARIMA(2,0,0)(0,0,0)[12] ---
  Fold 1: RMSE=113.17, MAE=96.63, MAPE=29.64%, SMAPE=25.34%, MSE=12808.21
  Fold 2: RMSE=410.78, MAE=353.06, MAPE=43.07%, SMAPE=57.52%, MSE=168736.57
  Fold 3: RMSE=289.61, MAE=237.85, MAPE=62.15%, SMAPE=40.65%, MSE=83874.34
  Fold 4: RMSE=310.67, MAE=262.22, MAPE=34.81%, SMAPE=43.60%, MSE=96516.09
  Promedio -> RMSE=281.06, MAE=237.44, MAPE=42.42%, SMAPE=41.78%, MSE=90483.80

--- SARIMA(2

,modelo,RMSE,MAE,MAPE,SMAPE,MSE
0,"SARIMA(1,0,0)(0,0,0)[12]",2.702400e+02,2.254300e+02,3.790000e+01,40.82,8.426076e+04
8,"SARIMA(1,0,0)(0,1,1)[12]",2.765800e+02,2.419400e+02,5.147000e+01,40.30,7.888096e+04
1,"SARIMA(2,0,0)(0,0,0)[12]",2.810600e+02,2.374400e+02,4.242000e+01,41.78,9.048380e+04
6,"SARIMA(0,0,1)(1,0,0)[12]",3.024100e+02,2.588300e+02,4.588000e+01,49.56,1.052862e+05
2,"SARIMA(2,0,0)(1,0,0)[12]",3.025600e+02,2.606300e+02,4.536000e+01,48.03,1.066951e+05
7,"SARIMA(2,0,0)(0,1,1)[12]",3.159900e+02,2.758900e+02,5.668000e+01,45.58,1.073692e+05
5,"SARIMA(1,0,1)(1,0,0)[12]",3.186500e+02,2.753200e+02,4.822000e+01,50.70,1.200631e+05
4,"SARIMA(2,0,0)(1,0,1)[12]",2.861332e+04,1.620434e+04,2.053820e+03,80.69,3.230275e+09
3,"SARIMA(2,0,0)(0,0,1)[12]",1.279341e+13,4.128322e+12,4.823904e+11,79.92,6.546849e+26


In [21]:
print("-" * 50)
print(f"Evaluación final en Test (2025) - {mejor_nombre}")
print("-" * 50)

modelo_final_sarima = SARIMAX(
    train_d04,
    order=mejor_order,
    seasonal_order=mejor_seasonal,
    enforce_stationarity=False,
    enforce_invertibility=False
)
modelo_final_sarima_ajustado = modelo_final_sarima.fit(disp=False)

pred_test_sarima = modelo_final_sarima_ajustado.forecast(steps=len(test_d04))

real_test = test_d04.values
pred_test = pred_test_sarima.values

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = real_test - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - SARIMA(1,0,0)(0,0,0)[12]
--------------------------------------------------
RMSE: 98.17
MAE: 77.92
MAPE: 37.39%
SMAPE: 26.72%
MSE: 9636.56


In [22]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_d04.index, y=train_d04.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_d04.index, y=test_d04.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_d04.index, y=pred_test_sarima.values,
    mode='lines+markers', name=f'Predicción {mejor_nombre}',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'D04 - {mejor_nombre}: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_d04.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_sarima.values,
    mode='lines+markers', name=f'Predicción {mejor_nombre}',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_d04.values - pred_test_sarima.values

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## Dynamic Optimized Theta

In [23]:
def evaluar_theta(train_fold, val_fold, theta=2.0, period=12):
    """
    Ajusta un modelo Theta con el parámetro theta dado.
    Procedimiento:
      1. Descomponer la serie con STL (period=12) para obtener estacionalidad.
      2. Serie desestacionalizada = serie - estacionalidad.
      3. Ajustar regresión lineal a la serie desestacionalizada (tendencia global).
      4. Ajustar SES a la serie desestacionalizada.
      5. Combinar pronósticos: Yhat = SES_forecast + theta*(trend_forecast - SES_forecast).
      6. Sumar estacionalidad (último año).
    Retorna predicciones, métricas y un diccionario con información del modelo.
    """
    if len(train_fold) < 2 * period:
        ses = ExponentialSmoothing(train_fold, trend=None, seasonal=None,
                                   initialization_method='estimated').fit()
        pred = ses.forecast(len(val_fold))
        real = val_fold.values
        pred = pred.values
        mse = mean_squared_error(real, pred)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(real, pred)
        mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
        smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
        return pred, {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'SMAPE': smape}, {'theta': theta, 'method': 'SES'}
    
    stl = STL(train_fold, period=period, robust=True).fit()
    seasonal = stl.seasonal
    deseasonalized = train_fold - seasonal

    x = np.arange(len(deseasonalized))
    slope, intercept, _, _, _ = sp_stats.linregress(x, deseasonalized.values)
    trend_line = slope * x + intercept

    ses_model = ExponentialSmoothing(deseasonalized, trend=None, seasonal=None,
                                     initialization_method='estimated').fit()
    
    h = len(val_fold)
    ses_forecast = ses_model.forecast(h)
    last_idx = len(deseasonalized) - 1
    trend_extrapolated = slope * np.arange(last_idx + 1, last_idx + 1 + h) + intercept

    theta_forecast = ses_forecast.values + theta * (trend_extrapolated - ses_forecast.values)

    last_year_seasonal = seasonal.iloc[-period:].values
    repeats = int(np.ceil(h / period))
    seasonal_pattern = np.tile(last_year_seasonal, repeats)[:h]
    final_forecast = theta_forecast + seasonal_pattern

    real = val_fold.values
    pred = final_forecast

    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'SMAPE': smape}
    modelo_info = {'theta': theta, 'slope': slope, 'intercept': intercept}
    
    return final_forecast, metricas, modelo_info

In [24]:
thetas = np.arange(0.0, 4.0 + 0.25, 0.25)

resultados_cv_theta = []

print("-" * 50)
print("Validación cruzada - Optimización de Theta (D04)")
print("-" * 50)

for theta in thetas:
    metricas_folds = {'RMSE': [], 'MAE': [], 'MAPE': [], 'SMAPE': [], 'MSE': []}
    
    for i, (train_fold, val_fold) in enumerate(ventanas_d04, 1):
        _, met, _ = evaluar_theta(train_fold, val_fold, theta=theta, period=12)
        metricas_folds['RMSE'].append(met['RMSE'])
        metricas_folds['MAE'].append(met['MAE'])
        metricas_folds['MAPE'].append(met['MAPE'])
        metricas_folds['SMAPE'].append(met['SMAPE'])
        metricas_folds['MSE'].append(met['MSE'])
    
    promedio = {k: np.mean(v) for k, v in metricas_folds.items()}
    resultados_cv_theta.append({
        'theta': theta,
        'RMSE': promedio['RMSE'],
        'MAE': promedio['MAE'],
        'MAPE': promedio['MAPE'],
        'SMAPE': promedio['SMAPE'],
        'MSE': promedio['MSE']
    })

df_cv_theta = pd.DataFrame(resultados_cv_theta)
mejor_idx = df_cv_theta['RMSE'].idxmin()
mejor_theta = df_cv_theta.loc[mejor_idx, 'theta']
mejor_rmse_cv = df_cv_theta.loc[mejor_idx, 'RMSE']

print("\n--- Resultados de optimización de theta ---")
print(f"Mejor theta: {mejor_theta:.2f} (RMSE CV promedio = {mejor_rmse_cv:.2f})")
print("\nTop 5 configuraciones:")
display(df_cv_theta.sort_values('RMSE').head())

print("\n" + "-" * 50)
print(f"Validación cruzada - Theta dinámico óptimo (θ={mejor_theta:.2f})")
print("-" * 50)

metricas_opt_folds = {'RMSE': [], 'MAE': [], 'MAPE': [], 'SMAPE': [], 'MSE': []}
for i, (train_fold, val_fold) in enumerate(ventanas_d04, 1):
    _, met, _ = evaluar_theta(train_fold, val_fold, theta=mejor_theta, period=12)
    metricas_opt_folds['RMSE'].append(met['RMSE'])
    metricas_opt_folds['MAE'].append(met['MAE'])
    metricas_opt_folds['MAPE'].append(met['MAPE'])
    metricas_opt_folds['SMAPE'].append(met['SMAPE'])
    metricas_opt_folds['MSE'].append(met['MSE'])
    print(f"  Fold {i}: RMSE={met['RMSE']:.2f}, MAE={met['MAE']:.2f}, "
          f"MAPE={met['MAPE']:.2f}%, SMAPE={met['SMAPE']:.2f}%, MSE={met['MSE']:.2f}")

prom_opt = {k: np.mean(v) for k, v in metricas_opt_folds.items()}
print(f"  Promedio -> RMSE={prom_opt['RMSE']:.2f}, MAE={prom_opt['MAE']:.2f}, "
      f"MAPE={prom_opt['MAPE']:.2f}%, SMAPE={prom_opt['SMAPE']:.2f}%, MSE={promedio['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - Optimización de Theta (D04)
--------------------------------------------------

--- Resultados de optimización de theta ---
Mejor theta: 1.00 (RMSE CV promedio = 290.10)

Top 5 configuraciones:


,theta,RMSE,MAE,MAPE,SMAPE,MSE
4,1.00,290.102350,244.828339,49.292939,43.556224,88579.146555
5,1.25,290.263644,246.821175,50.192174,43.851884,89141.184371
3,0.75,294.576122,248.861504,49.470983,44.270413,90913.609403
6,1.50,295.601489,252.013947,51.468477,44.481070,92599.722852
2,0.50,302.726018,257.278416,50.492608,45.709285,96144.572916



--------------------------------------------------
Validación cruzada - Theta dinámico óptimo (θ=1.00)
--------------------------------------------------
  Fold 1: RMSE=300.04, MAE=259.58, MAPE=80.06%, SMAPE=49.68%, MSE=90026.44
  Fold 2: RMSE=394.92, MAE=329.34, MAPE=40.04%, SMAPE=54.16%, MSE=155961.23
  Fold 3: RMSE=230.61, MAE=185.88, MAPE=43.16%, SMAPE=34.74%, MSE=53182.86
  Fold 4: RMSE=234.83, MAE=204.51, MAPE=33.92%, SMAPE=35.65%, MSE=55146.06
  Promedio -> RMSE=290.10, MAE=244.83, MAPE=49.29%, SMAPE=43.56%, MSE=286492.64


In [25]:
print("-" * 50)
print(f"Evaluación final en Test (2025) - Theta optimizado (θ={mejor_theta:.2f})")
print("-" * 50)

pred_test_theta, met_train, modelo_info = evaluar_theta(train_d04, test_d04, theta=mejor_theta, period=12)

real_test = test_d04.values
pred_test = pred_test_theta

mse_test = met_train['MSE']
rmse_test = met_train['RMSE']
mae_test = met_train['MAE']
mape_test = met_train['MAPE']
smape_test = met_train['SMAPE']

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = test_d04.values - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - Theta optimizado (θ=1.00)
--------------------------------------------------
RMSE: 390.45
MAE: 351.20
MAPE: 132.70%
SMAPE: 73.07%
MSE: 152450.62


In [26]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_d04.index, y=train_d04.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_d04.index, y=test_d04.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_d04.index, y=pred_test_theta,
    mode='lines+markers', name=f'Predicción Theta (θ={mejor_theta:.2f})',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'D04 - Theta Dinámico Optimizado (θ={mejor_theta:.2f}): Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_d04.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_theta,
    mode='lines+markers', name=f'Predicción Theta (θ={mejor_theta:.2f})',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_d04.values - pred_test_theta

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## Ridge

In [27]:
def crear_dataset_supervisado_ridge(serie_valores, fechas, n_input=12, n_output=12):
    """
    Crea X con rezagos y características determinísticas.
    Retorna X, y.
    """
    X, y = [], []
    for i in range(len(serie_valores) - n_input - n_output + 1):
        x_lags = serie_valores[i:i + n_input]
        fecha_fin_input = fechas[i + n_input - 1]
        tendencia = (i + n_input) / len(serie_valores)
        mes = fecha_fin_input.month
        mes_sin = np.sin(2 * np.pi * mes / 12)
        mes_cos = np.cos(2 * np.pi * mes / 12)
        fecha_primer_pred = fechas[i + n_input]
        covid = 1 if (fecha_primer_pred >= pd.Timestamp('2020-03-01') and 
                      fecha_primer_pred <= pd.Timestamp('2020-12-01')) else 0
        x_full = np.concatenate([x_lags, [tendencia, mes_sin, mes_cos, covid]])
        X.append(x_full)
        y.append(serie_valores[i + n_input:i + n_input + n_output])
    return np.array(X), np.array(y)

def evaluar_ridge(train_fold, val_fold, n_input=12, alphas=None):
    """
    Ajusta RidgeCV con características determinísticas.
    Retorna predicciones, métricas, modelo y scaler.
    """
    if alphas is None:
        alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
    
    X_train, y_train = crear_dataset_supervisado_ridge(
        train_fold.values, train_fold.index, n_input=n_input, n_output=len(val_fold)
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    tscv = TimeSeriesSplit(n_splits=3)
    modelo = RidgeCV(alphas=alphas, cv=tscv)
    modelo.fit(X_train_scaled, y_train)
    
    x_lags_pred = train_fold.values[-n_input:]
    fecha_fin_input_pred = train_fold.index[-1]
    tendencia_pred = 1.0
    mes_pred = fecha_fin_input_pred.month
    mes_sin_pred = np.sin(2 * np.pi * mes_pred / 12)
    mes_cos_pred = np.cos(2 * np.pi * mes_pred / 12)
    covid_pred = 1 if (val_fold.index[0] >= pd.Timestamp('2020-03-01') and 
                       val_fold.index[0] <= pd.Timestamp('2020-12-01')) else 0
    x_full_pred = np.concatenate([x_lags_pred, [tendencia_pred, mes_sin_pred, mes_cos_pred, covid_pred]])
    X_pred = x_full_pred.reshape(1, -1)
    X_pred_scaled = scaler.transform(X_pred)
    
    predicciones = modelo.predict(X_pred_scaled).flatten()
    
    real = val_fold.values
    pred = predicciones
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, modelo, scaler

In [28]:
resultados_cv_ridge = []

print("-" * 50)
print("Validación cruzada - Ridge Regression (D04)")
print("-" * 50)

for i, (train_fold, val_fold) in enumerate(ventanas_d04, 1):
    print(f"\n--- Fold {i} ---")
    
    predicciones, metricas, modelo, _ = evaluar_ridge(train_fold, val_fold)
    
    resultados_cv_ridge.append({
        'fold': i,
        'train_periodo': f"{train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}",
        'val_periodo': f"{val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}",
        'alpha': modelo.alpha_,
        'metricas': metricas
    })
    
    print(f"Train: {train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}")
    print(f"Val: {val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}")
    print(f"Alpha seleccionado: {modelo.alpha_:.2f}")
    print(f"RMSE: {metricas['RMSE']:.2f}")
    print(f"MAE: {metricas['MAE']:.2f}")
    print(f"MAPE: {metricas['MAPE']:.2f}%")
    print(f"SMAPE: {metricas['SMAPE']:.2f}%")
    print(f"MSE: {metricas['MSE']:.2f}")

metricas_promedio_ridge = {
    'RMSE': np.mean([r['metricas']['RMSE'] for r in resultados_cv_ridge]),
    'MAE': np.mean([r['metricas']['MAE'] for r in resultados_cv_ridge]),
    'MAPE': np.mean([r['metricas']['MAPE'] for r in resultados_cv_ridge]),
    'SMAPE': np.mean([r['metricas']['SMAPE'] for r in resultados_cv_ridge]),
    'MSE': np.mean([r['metricas']['MSE'] for r in resultados_cv_ridge])
}

print("\n" + "-" * 50)
print("Métricas promedio en validación cruzada")
print("-" * 50)
print(f"RMSE: {metricas_promedio_ridge['RMSE']:.2f}")
print(f"MAE: {metricas_promedio_ridge['MAE']:.2f}")
print(f"MAPE: {metricas_promedio_ridge['MAPE']:.2f}%")
print(f"SMAPE: {metricas_promedio_ridge['SMAPE']:.2f}%")
print(f"MSE: {metricas_promedio_ridge['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - Ridge Regression (D04)
--------------------------------------------------

--- Fold 1 ---
Train: 2018-01 - 2020-12
Val: 2021-01 - 2021-12
Alpha seleccionado: 10.00
RMSE: 236.20
MAE: 210.52
MAPE: 62.59%
SMAPE: 43.88%
MSE: 55788.70

--- Fold 2 ---
Train: 2018-01 - 2021-12
Val: 2022-01 - 2022-12
Alpha seleccionado: 10.00
RMSE: 399.49
MAE: 344.30
MAPE: 42.17%
SMAPE: 56.69%
MSE: 159591.17

--- Fold 3 ---
Train: 2018-01 - 2022-12
Val: 2023-01 - 2023-12
Alpha seleccionado: 10.00
RMSE: 193.84
MAE: 165.52
MAPE: 40.27%
SMAPE: 31.06%
MSE: 37572.46

--- Fold 4 ---
Train: 2018-01 - 2023-12
Val: 2024-01 - 2024-12
Alpha seleccionado: 10.00
RMSE: 243.84
MAE: 206.27
MAPE: 27.31%
SMAPE: 32.37%
MSE: 59455.70

--------------------------------------------------
Métricas promedio en validación cruzada
--------------------------------------------------
RMSE: 268.34
MAE: 231.65
MAPE: 43.09%
SMAPE: 41.00%
MSE: 78102.01


In [29]:
print("-" * 50)
print("Evaluación final en Test (2025) - Ridge Regression")
print("-" * 50)

X_train_full, y_train_full = crear_dataset_supervisado_ridge(
    train_d04.values, train_d04.index, n_input=12, n_output=12
)

scaler_final = StandardScaler()
X_train_full_scaled = scaler_final.fit_transform(X_train_full)

tscv = TimeSeriesSplit(n_splits=3)
alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
modelo_final_ridge = RidgeCV(alphas=alphas, cv=tscv)
modelo_final_ridge.fit(X_train_full_scaled, y_train_full)
print(f"Alpha seleccionado: {modelo_final_ridge.alpha_:.2f}")

x_lags_pred_test = train_d04.values[-12:]
fecha_fin_pred_test = train_d04.index[-1]
tendencia_pred_test = 1.0
mes_pred_test = fecha_fin_pred_test.month
mes_sin_pred_test = np.sin(2 * np.pi * mes_pred_test / 12)
mes_cos_pred_test = np.cos(2 * np.pi * mes_pred_test / 12)
covid_pred_test = 0
x_full_pred_test = np.concatenate([x_lags_pred_test, [tendencia_pred_test, mes_sin_pred_test, mes_cos_pred_test, covid_pred_test]])
X_pred_test = x_full_pred_test.reshape(1, -1)
X_pred_test_scaled = scaler_final.transform(X_pred_test)

pred_test_ridge = modelo_final_ridge.predict(X_pred_test_scaled).flatten()

real_test = test_d04.values
pred_test = pred_test_ridge

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = test_d04.values - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - Ridge Regression
--------------------------------------------------
Alpha seleccionado: 100.00
RMSE: 303.12
MAE: 288.11
MAPE: 132.22%
SMAPE: 66.56%
MSE: 91880.80


In [30]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_d04.index, y=train_d04.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_d04.index, y=test_d04.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_d04.index, y=pred_test_ridge,
    mode='lines+markers', name=f'Predicción Ridge (α={modelo_final_ridge.alpha_:.2f})',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'D04 - Ridge Regression: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_d04.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_ridge,
    mode='lines+markers', name=f'Predicción Ridge (α={modelo_final_ridge.alpha_:.2f})',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_d04.values - pred_test_ridge

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## Lasso

In [31]:
def crear_dataset_supervisado_lasso(serie_valores, fechas, n_input=12, n_output=12):
    """
    Crea X con rezagos y características determinísticas.
    Retorna X, y.
    """
    X, y = [], []
    for i in range(len(serie_valores) - n_input - n_output + 1):
        x_lags = serie_valores[i:i + n_input]
        fecha_fin_input = fechas[i + n_input - 1]
        tendencia = (i + n_input) / len(serie_valores)
        mes = fecha_fin_input.month
        mes_sin = np.sin(2 * np.pi * mes / 12)
        mes_cos = np.cos(2 * np.pi * mes / 12)
        fecha_primer_pred = fechas[i + n_input]
        covid = 1 if (fecha_primer_pred >= pd.Timestamp('2020-03-01') and 
                      fecha_primer_pred <= pd.Timestamp('2020-12-01')) else 0
        x_full = np.concatenate([x_lags, [tendencia, mes_sin, mes_cos, covid]])
        X.append(x_full)
        y.append(serie_valores[i + n_input:i + n_input + n_output])
    return np.array(X), np.array(y)

def evaluar_lasso(train_fold, val_fold, n_input=12, alphas=None):
    """
    Ajusta MultiTaskLassoCV con características determinísticas.
    Retorna predicciones, métricas, modelo y scaler.
    """
    if alphas is None:
        alphas = np.logspace(-4, 4, 50)
    
    X_train, y_train = crear_dataset_supervisado_lasso(
        train_fold.values, train_fold.index, n_input=n_input, n_output=len(val_fold)
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    tscv = TimeSeriesSplit(n_splits=3)
    modelo = MultiTaskLassoCV(alphas=alphas, cv=tscv, max_iter=20000, random_state=42)
    modelo.fit(X_train_scaled, y_train)
    
    x_lags_pred = train_fold.values[-n_input:]
    fecha_fin_input_pred = train_fold.index[-1]
    tendencia_pred = 1.0
    mes_pred = fecha_fin_input_pred.month
    mes_sin_pred = np.sin(2 * np.pi * mes_pred / 12)
    mes_cos_pred = np.cos(2 * np.pi * mes_pred / 12)
    covid_pred = 1 if (val_fold.index[0] >= pd.Timestamp('2020-03-01') and 
                       val_fold.index[0] <= pd.Timestamp('2020-12-01')) else 0
    x_full_pred = np.concatenate([x_lags_pred, [tendencia_pred, mes_sin_pred, mes_cos_pred, covid_pred]])
    X_pred = x_full_pred.reshape(1, -1)
    X_pred_scaled = scaler.transform(X_pred)
    
    predicciones = modelo.predict(X_pred_scaled).flatten()
    
    real = val_fold.values
    pred = predicciones
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, modelo, scaler

In [32]:
resultados_cv_lasso = []

print("-" * 50)
print("Validación cruzada - Lasso Regression (D04)")
print("-" * 50)

for i, (train_fold, val_fold) in enumerate(ventanas_d04, 1):
    print(f"\n--- Fold {i} ---")
    
    predicciones, metricas, modelo, _ = evaluar_lasso(train_fold, val_fold)
    
    resultados_cv_lasso.append({
        'fold': i,
        'train_periodo': f"{train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}",
        'val_periodo': f"{val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}",
        'alpha': modelo.alpha_,
        'metricas': metricas
    })
    
    print(f"Train: {train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}")
    print(f"Val: {val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}")
    print(f"Alpha seleccionado: {modelo.alpha_:.2f}")
    print(f"RMSE: {metricas['RMSE']:.2f}")
    print(f"MAE: {metricas['MAE']:.2f}")
    print(f"MAPE: {metricas['MAPE']:.2f}%")
    print(f"SMAPE: {metricas['SMAPE']:.2f}%")
    print(f"MSE: {metricas['MSE']:.2f}")

metricas_promedio_lasso = {
    'RMSE': np.mean([r['metricas']['RMSE'] for r in resultados_cv_lasso]),
    'MAE': np.mean([r['metricas']['MAE'] for r in resultados_cv_lasso]),
    'MAPE': np.mean([r['metricas']['MAPE'] for r in resultados_cv_lasso]),
    'SMAPE': np.mean([r['metricas']['SMAPE'] for r in resultados_cv_lasso]),
    'MSE': np.mean([r['metricas']['MSE'] for r in resultados_cv_lasso])
}

print("\n" + "-" * 50)
print("Métricas promedio en validación cruzada")
print("-" * 50)
print(f"RMSE: {metricas_promedio_lasso['RMSE']:.2f}")
print(f"MAE: {metricas_promedio_lasso['MAE']:.2f}")
print(f"MAPE: {metricas_promedio_lasso['MAPE']:.2f}%")
print(f"SMAPE: {metricas_promedio_lasso['SMAPE']:.2f}%")
print(f"MSE: {metricas_promedio_lasso['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - Lasso Regression (D04)
--------------------------------------------------

--- Fold 1 ---
Train: 2018-01 - 2020-12
Val: 2021-01 - 2021-12
Alpha seleccionado: 109.85
RMSE: 182.84
MAE: 160.38
MAPE: 50.34%
SMAPE: 36.67%
MSE: 33431.01

--- Fold 2 ---
Train: 2018-01 - 2021-12
Val: 2022-01 - 2022-12
Alpha seleccionado: 75.43
RMSE: 558.35
MAE: 511.98
MAPE: 66.06%
SMAPE: 101.50%
MSE: 311751.90

--- Fold 3 ---
Train: 2018-01 - 2022-12
Val: 2023-01 - 2023-12
Alpha seleccionado: 10000.00
RMSE: 206.54
MAE: 186.87
MAPE: 39.92%
SMAPE: 34.52%
MSE: 42660.02

--- Fold 4 ---
Train: 2018-01 - 2023-12
Val: 2024-01 - 2024-12
Alpha seleccionado: 51.79
RMSE: 231.94
MAE: 202.37
MAPE: 27.99%
SMAPE: 31.53%
MSE: 53798.34

--------------------------------------------------
Métricas promedio en validación cruzada
--------------------------------------------------
RMSE: 294.92
MAE: 265.40
MAPE: 46.08%
SMAPE: 51.05%
MSE: 110410.32


In [33]:
print("-" * 50)
print("Evaluación final en Test (2025) - Lasso Regression")
print("-" * 50)

X_train_full, y_train_full = crear_dataset_supervisado_lasso(
    train_d04.values, train_d04.index, n_input=12, n_output=12
)

scaler_final = StandardScaler()
X_train_full_scaled = scaler_final.fit_transform(X_train_full)

tscv = TimeSeriesSplit(n_splits=3)
alphas = np.logspace(-4, 4, 50)
modelo_final_lasso = MultiTaskLassoCV(alphas=alphas, cv=tscv, max_iter=20000, random_state=42)
modelo_final_lasso.fit(X_train_full_scaled, y_train_full)
print(f"Alpha seleccionado: {modelo_final_lasso.alpha_:.2f}")

x_lags_pred_test = train_d04.values[-12:]
fecha_fin_pred_test = train_d04.index[-1]
tendencia_pred_test = 1.0
mes_pred_test = fecha_fin_pred_test.month
mes_sin_pred_test = np.sin(2 * np.pi * mes_pred_test / 12)
mes_cos_pred_test = np.cos(2 * np.pi * mes_pred_test / 12)
covid_pred_test = 0
x_full_pred_test = np.concatenate([x_lags_pred_test, [tendencia_pred_test, mes_sin_pred_test, mes_cos_pred_test, covid_pred_test]])
X_pred_test = x_full_pred_test.reshape(1, -1)
X_pred_test_scaled = scaler_final.transform(X_pred_test)

pred_test_lasso = modelo_final_lasso.predict(X_pred_test_scaled).flatten()

real_test = test_d04.values
pred_test = pred_test_lasso

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = test_d04.values - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - Lasso Regression
--------------------------------------------------
Alpha seleccionado: 35.56
RMSE: 314.70
MAE: 294.83
MAPE: 133.33%
SMAPE: 67.07%
MSE: 99035.21


In [34]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_d04.index, y=train_d04.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_d04.index, y=test_d04.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_d04.index, y=pred_test_lasso,
    mode='lines+markers', name=f'Predicción Lasso (α={modelo_final_lasso.alpha_:.2f})',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'D04 - Lasso Regression: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_d04.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_lasso,
    mode='lines+markers', name=f'Predicción Lasso (α={modelo_final_lasso.alpha_:.2f})',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_d04.values - pred_test_lasso

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## Random Forest

In [35]:
def crear_dataset_supervisado_rf(serie_valores, fechas, n_input=12, n_output=12):
    """
    Crea X con rezagos y características determinísticas.
    Retorna X, y.
    """
    X, y = [], []
    for i in range(len(serie_valores) - n_input - n_output + 1):
        x_lags = serie_valores[i:i + n_input]
        fecha_fin_input = fechas[i + n_input - 1]
        tendencia = (i + n_input) / len(serie_valores)
        mes = fecha_fin_input.month
        mes_sin = np.sin(2 * np.pi * mes / 12)
        mes_cos = np.cos(2 * np.pi * mes / 12)
        fecha_primer_pred = fechas[i + n_input]
        covid = 1 if (fecha_primer_pred >= pd.Timestamp('2020-03-01') and 
                      fecha_primer_pred <= pd.Timestamp('2020-12-01')) else 0
        x_full = np.concatenate([x_lags, [tendencia, mes_sin, mes_cos, covid]])
        X.append(x_full)
        y.append(serie_valores[i + n_input:i + n_input + n_output])
    return np.array(X), np.array(y)

def evaluar_random_forest(train_fold, val_fold, n_input=12, param_grid=None):
    """
    Ajusta Random Forest multi-output con búsqueda de hiperparámetros.
    Retorna predicciones, métricas, mejor modelo y scaler.
    """
    if param_grid is None:
        param_grid = {
            'estimator__n_estimators': [100, 200, 300],
            'estimator__max_depth': [5, 10, 15, None],
            'estimator__min_samples_split': [2, 5, 10]
        }
    
    X_train, y_train = crear_dataset_supervisado_rf(
        train_fold.values, train_fold.index, n_input=n_input, n_output=len(val_fold)
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    rf = RandomForestRegressor(random_state=42)
    multi_rf = MultiOutputRegressor(rf)
    
    tscv = TimeSeriesSplit(n_splits=3)
    grid = GridSearchCV(multi_rf, param_grid, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1)
    grid.fit(X_train_scaled, y_train)
    
    mejor_modelo = grid.best_estimator_
    
    x_lags_pred = train_fold.values[-n_input:]
    fecha_fin_input_pred = train_fold.index[-1]
    tendencia_pred = 1.0
    mes_pred = fecha_fin_input_pred.month
    mes_sin_pred = np.sin(2 * np.pi * mes_pred / 12)
    mes_cos_pred = np.cos(2 * np.pi * mes_pred / 12)
    covid_pred = 1 if (val_fold.index[0] >= pd.Timestamp('2020-03-01') and 
                       val_fold.index[0] <= pd.Timestamp('2020-12-01')) else 0
    x_full_pred = np.concatenate([x_lags_pred, [tendencia_pred, mes_sin_pred, mes_cos_pred, covid_pred]])
    X_pred = x_full_pred.reshape(1, -1)
    X_pred_scaled = scaler.transform(X_pred)
    
    predicciones = mejor_modelo.predict(X_pred_scaled).flatten()
    
    real = val_fold.values
    pred = predicciones
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, mejor_modelo, scaler

In [36]:
resultados_cv_rf = []

print("-" * 50)
print("Validación cruzada - Random Forest (D04)")
print("-" * 50)

for i, (train_fold, val_fold) in enumerate(ventanas_d04, 1):
    print(f"\n--- Fold {i} ---")
    
    predicciones, metricas, modelo, _ = evaluar_random_forest(train_fold, val_fold)
    
    best_params = modelo.get_params()
    n_estimators = best_params['estimator__n_estimators']
    max_depth = best_params['estimator__max_depth']
    min_samples_split = best_params['estimator__min_samples_split']
    
    resultados_cv_rf.append({
        'fold': i,
        'train_periodo': f"{train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}",
        'val_periodo': f"{val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}",
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'min_samples_split': min_samples_split,
        'metricas': metricas
    })
    
    print(f"Train: {train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}")
    print(f"Val: {val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}")
    print(f"Mejores parámetros: n_estimators={n_estimators}, max_depth={max_depth}, min_samples_split={min_samples_split}")
    print(f"RMSE: {metricas['RMSE']:.2f}")
    print(f"MAE: {metricas['MAE']:.2f}")
    print(f"MAPE: {metricas['MAPE']:.2f}%")
    print(f"SMAPE: {metricas['SMAPE']:.2f}%")
    print(f"MSE: {metricas['MSE']:.2f}")

metricas_promedio_rf = {
    'RMSE': np.mean([r['metricas']['RMSE'] for r in resultados_cv_rf]),
    'MAE': np.mean([r['metricas']['MAE'] for r in resultados_cv_rf]),
    'MAPE': np.mean([r['metricas']['MAPE'] for r in resultados_cv_rf]),
    'SMAPE': np.mean([r['metricas']['SMAPE'] for r in resultados_cv_rf]),
    'MSE': np.mean([r['metricas']['MSE'] for r in resultados_cv_rf])
}

print("\n" + "-" * 50)
print("Métricas promedio en validación cruzada")
print("-" * 50)
print(f"RMSE: {metricas_promedio_rf['RMSE']:.2f}")
print(f"MAE: {metricas_promedio_rf['MAE']:.2f}")
print(f"MAPE: {metricas_promedio_rf['MAPE']:.2f}%")
print(f"SMAPE: {metricas_promedio_rf['SMAPE']:.2f}%")
print(f"MSE: {metricas_promedio_rf['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - Random Forest (D04)
--------------------------------------------------

--- Fold 1 ---
Train: 2018-01 - 2020-12
Val: 2021-01 - 2021-12
Mejores parámetros: n_estimators=100, max_depth=5, min_samples_split=5
RMSE: 273.18
MAE: 250.83
MAPE: 74.75%
SMAPE: 50.35%
MSE: 74626.22

--- Fold 2 ---
Train: 2018-01 - 2021-12
Val: 2022-01 - 2022-12
Mejores parámetros: n_estimators=100, max_depth=10, min_samples_split=5
RMSE: 378.12
MAE: 315.76
MAPE: 38.06%
SMAPE: 49.95%
MSE: 142978.33

--- Fold 3 ---
Train: 2018-01 - 2022-12
Val: 2023-01 - 2023-12
Mejores parámetros: n_estimators=200, max_depth=10, min_samples_split=2
RMSE: 209.94
MAE: 192.52
MAPE: 41.76%
SMAPE: 35.28%
MSE: 44075.60

--- Fold 4 ---
Train: 2018-01 - 2023-12
Val: 2024-01 - 2024-12
Mejores parámetros: n_estimators=100, max_depth=10, min_samples_split=2
RMSE: 228.70
MAE: 205.21
MAPE: 29.35%
SMAPE: 32.32%
MSE: 52304.86

------------------------------------------------

In [37]:
print("-" * 50)
print("Evaluación final en Test (2025) - Random Forest")
print("-" * 50)

X_train_full, y_train_full = crear_dataset_supervisado_rf(
    train_d04.values, train_d04.index, n_input=12, n_output=12
)

scaler_final = StandardScaler()
X_train_full_scaled = scaler_final.fit_transform(X_train_full)

param_grid_final = {
    'estimator__n_estimators': [100, 200, 300, 400],
    'estimator__max_depth': [5, 10, 15, None],
    'estimator__min_samples_split': [2, 5, 10]
}

rf_final = RandomForestRegressor(random_state=42)
multi_rf_final = MultiOutputRegressor(rf_final)
tscv = TimeSeriesSplit(n_splits=3)
grid_final = GridSearchCV(multi_rf_final, param_grid_final, cv=tscv, 
                          scoring='neg_mean_squared_error', n_jobs=-1)
grid_final.fit(X_train_full_scaled, y_train_full)

modelo_final_rf = grid_final.best_estimator_
best_params_final = grid_final.best_params_
print(f"Mejores parámetros: {best_params_final}")

x_lags_pred_test = train_d04.values[-12:]
fecha_fin_pred_test = train_d04.index[-1]
tendencia_pred_test = 1.0
mes_pred_test = fecha_fin_pred_test.month
mes_sin_pred_test = np.sin(2 * np.pi * mes_pred_test / 12)
mes_cos_pred_test = np.cos(2 * np.pi * mes_pred_test / 12)
covid_pred_test = 0
x_full_pred_test = np.concatenate([x_lags_pred_test, [tendencia_pred_test, mes_sin_pred_test, mes_cos_pred_test, covid_pred_test]])
X_pred_test = x_full_pred_test.reshape(1, -1)
X_pred_test_scaled = scaler_final.transform(X_pred_test)

pred_test_rf = modelo_final_rf.predict(X_pred_test_scaled).flatten()

real_test = test_d04.values
pred_test = pred_test_rf

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = test_d04.values - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - Random Forest
--------------------------------------------------
Mejores parámetros: {'estimator__max_depth': 5, 'estimator__min_samples_split': 2, 'estimator__n_estimators': 200}
RMSE: 331.27
MAE: 310.22
MAPE: 141.30%
SMAPE: 69.05%
MSE: 109739.14


In [38]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_d04.index, y=train_d04.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_d04.index, y=test_d04.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_d04.index, y=pred_test_rf,
    mode='lines+markers', name='Predicción Random Forest',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'D04 - Random Forest: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_d04.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_rf,
    mode='lines+markers', name='Predicción Random Forest',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_d04.values - pred_test_rf

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## XGBoost

In [39]:
def crear_dataset_supervisado_xgb(serie_valores, fechas, n_input=12, n_output=12):
    """
    Crea X con rezagos y características determinísticas.
    Retorna X, y.
    """
    X, y = [], []
    for i in range(len(serie_valores) - n_input - n_output + 1):
        x_lags = serie_valores[i:i + n_input]
        fecha_fin_input = fechas[i + n_input - 1]
        tendencia = (i + n_input) / len(serie_valores)
        mes = fecha_fin_input.month
        mes_sin = np.sin(2 * np.pi * mes / 12)
        mes_cos = np.cos(2 * np.pi * mes / 12)
        fecha_primer_pred = fechas[i + n_input]
        covid = 1 if (fecha_primer_pred >= pd.Timestamp('2020-03-01') and 
                      fecha_primer_pred <= pd.Timestamp('2020-12-01')) else 0
        x_full = np.concatenate([x_lags, [tendencia, mes_sin, mes_cos, covid]])
        X.append(x_full)
        y.append(serie_valores[i + n_input:i + n_input + n_output])
    return np.array(X), np.array(y)

def evaluar_xgboost(train_fold, val_fold, n_input=12, param_grid=None):
    """
    Ajusta XGBoost multi-output con búsqueda de hiperparámetros.
    Retorna predicciones, métricas, mejor modelo y scaler.
    """
    if param_grid is None:
        param_grid = {
            'estimator__n_estimators': [100, 200, 300],
            'estimator__max_depth': [3, 5, 7],
            'estimator__learning_rate': [0.01, 0.05, 0.1],
            'estimator__subsample': [0.8, 1.0],
            'estimator__colsample_bytree': [0.8, 1.0]
        }
    
    X_train, y_train = crear_dataset_supervisado_xgb(
        train_fold.values, train_fold.index, n_input=n_input, n_output=len(val_fold)
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    xgb = XGBRegressor(random_state=42, objective='reg:squarederror', verbosity=0)
    multi_xgb = MultiOutputRegressor(xgb)
    
    tscv = TimeSeriesSplit(n_splits=3)
    grid = GridSearchCV(multi_xgb, param_grid, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1)
    grid.fit(X_train_scaled, y_train)
    
    mejor_modelo = grid.best_estimator_
    
    x_lags_pred = train_fold.values[-n_input:]
    fecha_fin_input_pred = train_fold.index[-1]
    tendencia_pred = 1.0
    mes_pred = fecha_fin_input_pred.month
    mes_sin_pred = np.sin(2 * np.pi * mes_pred / 12)
    mes_cos_pred = np.cos(2 * np.pi * mes_pred / 12)
    covid_pred = 1 if (val_fold.index[0] >= pd.Timestamp('2020-03-01') and 
                       val_fold.index[0] <= pd.Timestamp('2020-12-01')) else 0
    x_full_pred = np.concatenate([x_lags_pred, [tendencia_pred, mes_sin_pred, mes_cos_pred, covid_pred]])
    X_pred = x_full_pred.reshape(1, -1)
    X_pred_scaled = scaler.transform(X_pred)
    
    predicciones = mejor_modelo.predict(X_pred_scaled).flatten()
    
    real = val_fold.values
    pred = predicciones
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, mejor_modelo, scaler

In [40]:
resultados_cv_xgb = []

print("-" * 50)
print("Validación cruzada - XGBoost (D04)")
print("-" * 50)

for i, (train_fold, val_fold) in enumerate(ventanas_d04, 1):
    print(f"\n--- Fold {i} ---")
    
    predicciones, metricas, modelo, _ = evaluar_xgboost(train_fold, val_fold)
    
    best_params = modelo.get_params()
    n_estimators = best_params['estimator__n_estimators']
    max_depth = best_params['estimator__max_depth']
    learning_rate = best_params['estimator__learning_rate']
    subsample = best_params['estimator__subsample']
    colsample_bytree = best_params['estimator__colsample_bytree']
    
    resultados_cv_xgb.append({
        'fold': i,
        'train_periodo': f"{train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}",
        'val_periodo': f"{val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}",
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'learning_rate': learning_rate,
        'subsample': subsample,
        'colsample_bytree': colsample_bytree,
        'metricas': metricas
    })
    
    print(f"Train: {train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}")
    print(f"Val: {val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}")
    print(f"Mejores parámetros: n_estimators={n_estimators}, max_depth={max_depth}, learning_rate={learning_rate}, subsample={subsample}, colsample_bytree={colsample_bytree}")
    print(f"RMSE: {metricas['RMSE']:.2f}")
    print(f"MAE: {metricas['MAE']:.2f}")
    print(f"MAPE: {metricas['MAPE']:.2f}%")
    print(f"SMAPE: {metricas['SMAPE']:.2f}%")
    print(f"MSE: {metricas['MSE']:.2f}")

metricas_promedio_xgb = {
    'RMSE': np.mean([r['metricas']['RMSE'] for r in resultados_cv_xgb]),
    'MAE': np.mean([r['metricas']['MAE'] for r in resultados_cv_xgb]),
    'MAPE': np.mean([r['metricas']['MAPE'] for r in resultados_cv_xgb]),
    'SMAPE': np.mean([r['metricas']['SMAPE'] for r in resultados_cv_xgb]),
    'MSE': np.mean([r['metricas']['MSE'] for r in resultados_cv_xgb])
}

print("\n" + "-" * 50)
print("Métricas promedio en validación cruzada")
print("-" * 50)
print(f"RMSE: {metricas_promedio_xgb['RMSE']:.2f}")
print(f"MAE: {metricas_promedio_xgb['MAE']:.2f}")
print(f"MAPE: {metricas_promedio_xgb['MAPE']:.2f}%")
print(f"SMAPE: {metricas_promedio_xgb['SMAPE']:.2f}%")
print(f"MSE: {metricas_promedio_xgb['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - XGBoost (D04)
--------------------------------------------------

--- Fold 1 ---
Train: 2018-01 - 2020-12
Val: 2021-01 - 2021-12
Mejores parámetros: n_estimators=100, max_depth=5, learning_rate=0.01, subsample=0.8, colsample_bytree=1.0
RMSE: 346.90
MAE: 323.75
MAPE: 98.00%
SMAPE: 60.32%
MSE: 120340.54

--- Fold 2 ---
Train: 2018-01 - 2021-12
Val: 2022-01 - 2022-12
Mejores parámetros: n_estimators=100, max_depth=7, learning_rate=0.01, subsample=0.8, colsample_bytree=0.8
RMSE: 285.00
MAE: 242.96
MAPE: 30.61%
SMAPE: 35.65%
MSE: 81226.24

--- Fold 3 ---
Train: 2018-01 - 2022-12
Val: 2023-01 - 2023-12
Mejores parámetros: n_estimators=100, max_depth=3, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8
RMSE: 216.34
MAE: 194.53
MAPE: 43.46%
SMAPE: 35.67%
MSE: 46803.34

--- Fold 4 ---
Train: 2018-01 - 2023-12
Val: 2024-01 - 2024-12
Mejores parámetros: n_estimators=100, max_depth=7, learning_rate=0.01, subsample=0.8, co

In [41]:
print("-" * 50)
print("Evaluación final en Test (2025) - XGBoost")
print("-" * 50)

X_train_full, y_train_full = crear_dataset_supervisado_xgb(
    train_d04.values, train_d04.index, n_input=12, n_output=12
)

scaler_final = StandardScaler()
X_train_full_scaled = scaler_final.fit_transform(X_train_full)

param_grid_final = {
    'estimator__n_estimators': [100, 200, 300, 400],
    'estimator__max_depth': [3, 5, 7, 9],
    'estimator__learning_rate': [0.01, 0.05, 0.1],
    'estimator__subsample': [0.8, 1.0],
    'estimator__colsample_bytree': [0.8, 1.0]
}

xgb_final = XGBRegressor(random_state=42, objective='reg:squarederror', verbosity=0)
multi_xgb_final = MultiOutputRegressor(xgb_final)
tscv = TimeSeriesSplit(n_splits=3)
grid_final = GridSearchCV(multi_xgb_final, param_grid_final, cv=tscv, 
                          scoring='neg_mean_squared_error', n_jobs=-1)
grid_final.fit(X_train_full_scaled, y_train_full)

modelo_final_xgb = grid_final.best_estimator_
best_params_final = grid_final.best_params_
print(f"Mejores parámetros: {best_params_final}")

x_lags_pred_test = train_d04.values[-12:]
fecha_fin_pred_test = train_d04.index[-1]
tendencia_pred_test = 1.0
mes_pred_test = fecha_fin_pred_test.month
mes_sin_pred_test = np.sin(2 * np.pi * mes_pred_test / 12)
mes_cos_pred_test = np.cos(2 * np.pi * mes_pred_test / 12)
covid_pred_test = 0
x_full_pred_test = np.concatenate([x_lags_pred_test, [tendencia_pred_test, mes_sin_pred_test, mes_cos_pred_test, covid_pred_test]])
X_pred_test = x_full_pred_test.reshape(1, -1)
X_pred_test_scaled = scaler_final.transform(X_pred_test)

pred_test_xgb = modelo_final_xgb.predict(X_pred_test_scaled).flatten()

real_test = test_d04.values
pred_test = pred_test_xgb

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = test_d04.values - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - XGBoost
--------------------------------------------------
Mejores parámetros: {'estimator__colsample_bytree': 0.8, 'estimator__learning_rate': 0.01, 'estimator__max_depth': 9, 'estimator__n_estimators': 100, 'estimator__subsample': 0.8}
RMSE: 312.57
MAE: 295.66
MAPE: 137.06%
SMAPE: 67.43%
MSE: 97697.70


In [42]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_d04.index, y=train_d04.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_d04.index, y=test_d04.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_d04.index, y=pred_test_xgb,
    mode='lines+markers', name='Predicción XGBoost',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'D04 - XGBoost: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_d04.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_xgb,
    mode='lines+markers', name='Predicción XGBoost',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_d04.values - pred_test_xgb

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## LightGBM

In [43]:
def crear_dataset_supervisado_lgb(serie_valores, fechas, n_input=12, n_output=12):
    """
    Crea X con rezagos y características determinísticas.
    Retorna X, y.
    """
    X, y = [], []
    for i in range(len(serie_valores) - n_input - n_output + 1):
        x_lags = serie_valores[i:i + n_input]
        fecha_fin_input = fechas[i + n_input - 1]
        tendencia = (i + n_input) / len(serie_valores)
        mes = fecha_fin_input.month
        mes_sin = np.sin(2 * np.pi * mes / 12)
        mes_cos = np.cos(2 * np.pi * mes / 12)
        fecha_primer_pred = fechas[i + n_input]
        covid = 1 if (fecha_primer_pred >= pd.Timestamp('2020-03-01') and 
                      fecha_primer_pred <= pd.Timestamp('2020-12-01')) else 0
        x_full = np.concatenate([x_lags, [tendencia, mes_sin, mes_cos, covid]])
        X.append(x_full)
        y.append(serie_valores[i + n_input:i + n_input + n_output])
    return np.array(X), np.array(y)

def evaluar_lightgbm(train_fold, val_fold, n_input=12, param_grid=None):
    """
    Ajusta LightGBM multi-output con búsqueda de hiperparámetros.
    Retorna predicciones, métricas, mejor modelo y scaler.
    """
    if param_grid is None:
        param_grid = {
            'estimator__n_estimators': [100, 200, 300],
            'estimator__max_depth': [3, 5, 7, -1],
            'estimator__learning_rate': [0.01, 0.05, 0.1],
            'estimator__subsample': [0.8, 1.0],
            'estimator__colsample_bytree': [0.8, 1.0]
        }
    
    X_train, y_train = crear_dataset_supervisado_lgb(
        train_fold.values, train_fold.index, n_input=n_input, n_output=len(val_fold)
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    lgb = LGBMRegressor(random_state=42, verbosity=-1)
    multi_lgb = MultiOutputRegressor(lgb)
    
    tscv = TimeSeriesSplit(n_splits=3)
    grid = GridSearchCV(multi_lgb, param_grid, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1)
    grid.fit(X_train_scaled, y_train)
    
    mejor_modelo = grid.best_estimator_
    
    x_lags_pred = train_fold.values[-n_input:]
    fecha_fin_input_pred = train_fold.index[-1]
    tendencia_pred = 1.0
    mes_pred = fecha_fin_input_pred.month
    mes_sin_pred = np.sin(2 * np.pi * mes_pred / 12)
    mes_cos_pred = np.cos(2 * np.pi * mes_pred / 12)
    covid_pred = 1 if (val_fold.index[0] >= pd.Timestamp('2020-03-01') and 
                       val_fold.index[0] <= pd.Timestamp('2020-12-01')) else 0
    x_full_pred = np.concatenate([x_lags_pred, [tendencia_pred, mes_sin_pred, mes_cos_pred, covid_pred]])
    X_pred = x_full_pred.reshape(1, -1)
    X_pred_scaled = scaler.transform(X_pred)
    
    predicciones = mejor_modelo.predict(X_pred_scaled).flatten()
    
    real = val_fold.values
    pred = predicciones
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, mejor_modelo, scaler

In [44]:
resultados_cv_lgb = []

print("-" * 50)
print("Validación cruzada - LightGBM (D04)")
print("-" * 50)

for i, (train_fold, val_fold) in enumerate(ventanas_d04, 1):
    print(f"\n--- Fold {i} ---")
    
    predicciones, metricas, modelo, _ = evaluar_lightgbm(train_fold, val_fold)
    
    best_params = modelo.get_params()
    n_estimators = best_params['estimator__n_estimators']
    max_depth = best_params['estimator__max_depth']
    learning_rate = best_params['estimator__learning_rate']
    subsample = best_params['estimator__subsample']
    colsample_bytree = best_params['estimator__colsample_bytree']
    
    resultados_cv_lgb.append({
        'fold': i,
        'train_periodo': f"{train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}",
        'val_periodo': f"{val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}",
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'learning_rate': learning_rate,
        'subsample': subsample,
        'colsample_bytree': colsample_bytree,
        'metricas': metricas
    })
    
    print(f"Train: {train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}")
    print(f"Val: {val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}")
    print(f"Mejores parámetros: n_estimators={n_estimators}, max_depth={max_depth}, learning_rate={learning_rate}, subsample={subsample}, colsample_bytree={colsample_bytree}")
    print(f"RMSE: {metricas['RMSE']:.2f}")
    print(f"MAE: {metricas['MAE']:.2f}")
    print(f"MAPE: {metricas['MAPE']:.2f}%")
    print(f"SMAPE: {metricas['SMAPE']:.2f}%")
    print(f"MSE: {metricas['MSE']:.2f}")

metricas_promedio_lgb = {
    'RMSE': np.mean([r['metricas']['RMSE'] for r in resultados_cv_lgb]),
    'MAE': np.mean([r['metricas']['MAE'] for r in resultados_cv_lgb]),
    'MAPE': np.mean([r['metricas']['MAPE'] for r in resultados_cv_lgb]),
    'SMAPE': np.mean([r['metricas']['SMAPE'] for r in resultados_cv_lgb]),
    'MSE': np.mean([r['metricas']['MSE'] for r in resultados_cv_lgb])
}

print("\n" + "-" * 50)
print("Métricas promedio en validación cruzada")
print("-" * 50)
print(f"RMSE: {metricas_promedio_lgb['RMSE']:.2f}")
print(f"MAE: {metricas_promedio_lgb['MAE']:.2f}")
print(f"MAPE: {metricas_promedio_lgb['MAPE']:.2f}%")
print(f"SMAPE: {metricas_promedio_lgb['SMAPE']:.2f}%")
print(f"MSE: {metricas_promedio_lgb['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - LightGBM (D04)
--------------------------------------------------

--- Fold 1 ---
Train: 2018-01 - 2020-12
Val: 2021-01 - 2021-12
Mejores parámetros: n_estimators=100, max_depth=3, learning_rate=0.01, subsample=0.8, colsample_bytree=0.8
RMSE: 315.55
MAE: 287.38
MAPE: 89.79%
SMAPE: 55.63%
MSE: 99572.22

--- Fold 2 ---
Train: 2018-01 - 2021-12
Val: 2022-01 - 2022-12
Mejores parámetros: n_estimators=100, max_depth=3, learning_rate=0.01, subsample=0.8, colsample_bytree=0.8
RMSE: 298.92
MAE: 259.47
MAPE: 34.70%
SMAPE: 38.22%
MSE: 89351.93

--- Fold 3 ---
Train: 2018-01 - 2022-12
Val: 2023-01 - 2023-12
Mejores parámetros: n_estimators=100, max_depth=3, learning_rate=0.01, subsample=0.8, colsample_bytree=0.8
RMSE: 206.54
MAE: 186.87
MAPE: 39.92%
SMAPE: 34.52%
MSE: 42660.02

--- Fold 4 ---
Train: 2018-01 - 2023-12
Val: 2024-01 - 2024-12
Mejores parámetros: n_estimators=100, max_depth=3, learning_rate=0.01, subsample=0.8, c

In [45]:
print("-" * 50)
print("Evaluación final en Test (2025) - LightGBM")
print("-" * 50)

X_train_full, y_train_full = crear_dataset_supervisado_lgb(
    train_d04.values, train_d04.index, n_input=12, n_output=12
)

scaler_final = StandardScaler()
X_train_full_scaled = scaler_final.fit_transform(X_train_full)

param_grid_final = {
    'estimator__n_estimators': [100, 200, 300, 400],
    'estimator__max_depth': [3, 5, 7, -1],
    'estimator__learning_rate': [0.01, 0.05, 0.1],
    'estimator__subsample': [0.8, 1.0],
    'estimator__colsample_bytree': [0.8, 1.0]
}

lgb_final = LGBMRegressor(random_state=42, verbosity=-1)
multi_lgb_final = MultiOutputRegressor(lgb_final)
tscv = TimeSeriesSplit(n_splits=3)
grid_final = GridSearchCV(multi_lgb_final, param_grid_final, cv=tscv, 
                          scoring='neg_mean_squared_error', n_jobs=-1)
grid_final.fit(X_train_full_scaled, y_train_full)

modelo_final_lgb = grid_final.best_estimator_
best_params_final = grid_final.best_params_
print(f"Mejores parámetros: {best_params_final}")

x_lags_pred_test = train_d04.values[-12:]
fecha_fin_pred_test = train_d04.index[-1]
tendencia_pred_test = 1.0
mes_pred_test = fecha_fin_pred_test.month
mes_sin_pred_test = np.sin(2 * np.pi * mes_pred_test / 12)
mes_cos_pred_test = np.cos(2 * np.pi * mes_pred_test / 12)
covid_pred_test = 0
x_full_pred_test = np.concatenate([x_lags_pred_test, [tendencia_pred_test, mes_sin_pred_test, mes_cos_pred_test, covid_pred_test]])
X_pred_test = x_full_pred_test.reshape(1, -1)
X_pred_test_scaled = scaler_final.transform(X_pred_test)

pred_test_lgb = modelo_final_lgb.predict(X_pred_test_scaled).flatten()

real_test = test_d04.values
pred_test = pred_test_lgb

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = test_d04.values - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - LightGBM
--------------------------------------------------
Mejores parámetros: {'estimator__colsample_bytree': 0.8, 'estimator__learning_rate': 0.01, 'estimator__max_depth': 3, 'estimator__n_estimators': 100, 'estimator__subsample': 0.8}
RMSE: 297.17
MAE: 279.47
MAPE: 130.98%
SMAPE: 65.08%
MSE: 88308.78


In [46]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_d04.index, y=train_d04.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_d04.index, y=test_d04.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_d04.index, y=pred_test_lgb,
    mode='lines+markers', name='Predicción LightGBM',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'D04 - LightGBM: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_d04.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_lgb,
    mode='lines+markers', name='Predicción LightGBM',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_d04.values - pred_test_lgb

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## KNN

In [47]:
def crear_dataset_supervisado_knn(serie_valores, fechas, n_input=12, n_output=12):
    """
    Crea X con rezagos y características determinísticas.
    Retorna X, y.
    """
    X, y = [], []
    for i in range(len(serie_valores) - n_input - n_output + 1):
        x_lags = serie_valores[i:i + n_input]
        fecha_fin_input = fechas[i + n_input - 1]
        tendencia = (i + n_input) / len(serie_valores)
        mes = fecha_fin_input.month
        mes_sin = np.sin(2 * np.pi * mes / 12)
        mes_cos = np.cos(2 * np.pi * mes / 12)
        fecha_primer_pred = fechas[i + n_input]
        covid = 1 if (fecha_primer_pred >= pd.Timestamp('2020-03-01') and 
                      fecha_primer_pred <= pd.Timestamp('2020-12-01')) else 0
        x_full = np.concatenate([x_lags, [tendencia, mes_sin, mes_cos, covid]])
        X.append(x_full)
        y.append(serie_valores[i + n_input:i + n_input + n_output])
    return np.array(X), np.array(y)

def evaluar_knn(train_fold, val_fold, n_input=12, param_grid=None):
    """
    Ajusta KNN multi‑output con búsqueda de hiperparámetros.
    Retorna predicciones, métricas, mejor modelo y scaler.
    """
    if param_grid is None:
        param_grid = {
            'estimator__n_neighbors': [2, 3, 5, 7, 10],
            'estimator__weights': ['uniform', 'distance'],
            'estimator__p': [1, 2]   # 1 = Manhattan, 2 = Euclidiana
        }
    
    X_train, y_train = crear_dataset_supervisado_knn(
        train_fold.values, train_fold.index, n_input=n_input, n_output=len(val_fold)
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    knn = KNeighborsRegressor()
    multi_knn = MultiOutputRegressor(knn)
    
    tscv = TimeSeriesSplit(n_splits=3)
    grid = GridSearchCV(multi_knn, param_grid, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1)
    grid.fit(X_train_scaled, y_train)
    
    mejor_modelo = grid.best_estimator_
    
    x_lags_pred = train_fold.values[-n_input:]
    fecha_fin_input_pred = train_fold.index[-1]
    tendencia_pred = 1.0
    mes_pred = fecha_fin_input_pred.month
    mes_sin_pred = np.sin(2 * np.pi * mes_pred / 12)
    mes_cos_pred = np.cos(2 * np.pi * mes_pred / 12)
    covid_pred = 1 if (val_fold.index[0] >= pd.Timestamp('2020-03-01') and 
                       val_fold.index[0] <= pd.Timestamp('2020-12-01')) else 0
    x_full_pred = np.concatenate([x_lags_pred, [tendencia_pred, mes_sin_pred, mes_cos_pred, covid_pred]])
    X_pred = x_full_pred.reshape(1, -1)
    X_pred_scaled = scaler.transform(X_pred)
    
    predicciones = mejor_modelo.predict(X_pred_scaled).flatten()
    
    real = val_fold.values
    pred = predicciones
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, mejor_modelo, scaler

In [48]:
resultados_cv_knn = []

print("-" * 50)
print("Validación cruzada - KNN Regression (D04)")
print("-" * 50)

for i, (train_fold, val_fold) in enumerate(ventanas_d04, 1):
    print(f"\n--- Fold {i} ---")
    
    predicciones, metricas, modelo, _ = evaluar_knn(train_fold, val_fold)
    
    best_params = modelo.get_params()
    n_neighbors = best_params['estimator__n_neighbors']
    weights = best_params['estimator__weights']
    p = best_params['estimator__p']
    
    resultados_cv_knn.append({
        'fold': i,
        'train_periodo': f"{train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}",
        'val_periodo': f"{val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}",
        'n_neighbors': n_neighbors,
        'weights': weights,
        'p': p,
        'metricas': metricas
    })
    
    print(f"Train: {train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}")
    print(f"Val: {val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}")
    print(f"Mejores parámetros: n_neighbors={n_neighbors}, weights={weights}, p={p}")
    print(f"RMSE: {metricas['RMSE']:.2f}")
    print(f"MAE: {metricas['MAE']:.2f}")
    print(f"MAPE: {metricas['MAPE']:.2f}%")
    print(f"SMAPE: {metricas['SMAPE']:.2f}%")
    print(f"MSE: {metricas['MSE']:.2f}")

metricas_promedio_knn = {
    'RMSE': np.mean([r['metricas']['RMSE'] for r in resultados_cv_knn]),
    'MAE': np.mean([r['metricas']['MAE'] for r in resultados_cv_knn]),
    'MAPE': np.mean([r['metricas']['MAPE'] for r in resultados_cv_knn]),
    'SMAPE': np.mean([r['metricas']['SMAPE'] for r in resultados_cv_knn]),
    'MSE': np.mean([r['metricas']['MSE'] for r in resultados_cv_knn])
}

print("\n" + "-" * 50)
print("Métricas promedio en validación cruzada")
print("-" * 50)
print(f"RMSE: {metricas_promedio_knn['RMSE']:.2f}")
print(f"MAE: {metricas_promedio_knn['MAE']:.2f}")
print(f"MAPE: {metricas_promedio_knn['MAPE']:.2f}%")
print(f"SMAPE: {metricas_promedio_knn['SMAPE']:.2f}%")
print(f"MSE: {metricas_promedio_knn['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - KNN Regression (D04)
--------------------------------------------------

--- Fold 1 ---
Train: 2018-01 - 2020-12
Val: 2021-01 - 2021-12
Mejores parámetros: n_neighbors=3, weights=uniform, p=1
RMSE: 226.28
MAE: 197.36
MAPE: 62.54%
SMAPE: 42.35%
MSE: 51201.14

--- Fold 2 ---
Train: 2018-01 - 2021-12
Val: 2022-01 - 2022-12
Mejores parámetros: n_neighbors=3, weights=distance, p=1
RMSE: 309.86
MAE: 266.12
MAPE: 33.41%
SMAPE: 40.17%
MSE: 96013.36

--- Fold 3 ---
Train: 2018-01 - 2022-12
Val: 2023-01 - 2023-12
Mejores parámetros: n_neighbors=5, weights=distance, p=1
RMSE: 196.05
MAE: 174.52
MAPE: 36.05%
SMAPE: 32.19%
MSE: 38437.16

--- Fold 4 ---
Train: 2018-01 - 2023-12
Val: 2024-01 - 2024-12
Mejores parámetros: n_neighbors=5, weights=distance, p=1
RMSE: 244.68
MAE: 210.08
MAPE: 29.85%
SMAPE: 33.23%
MSE: 59866.32

--------------------------------------------------
Métricas promedio en validación cruzada
-----------------

In [49]:
print("-" * 50)
print("Evaluación final en Test (2025) - KNN Regression")
print("-" * 50)

X_train_full, y_train_full = crear_dataset_supervisado_knn(
    train_d04.values, train_d04.index, n_input=12, n_output=12
)

scaler_final = StandardScaler()
X_train_full_scaled = scaler_final.fit_transform(X_train_full)

param_grid_final = {
    'estimator__n_neighbors': [2, 3, 5, 7, 10, 12],
    'estimator__weights': ['uniform', 'distance'],
    'estimator__p': [1, 2]
}

knn_final = KNeighborsRegressor()
multi_knn_final = MultiOutputRegressor(knn_final)
tscv = TimeSeriesSplit(n_splits=3)
grid_final = GridSearchCV(multi_knn_final, param_grid_final, cv=tscv, 
                          scoring='neg_mean_squared_error', n_jobs=-1)
grid_final.fit(X_train_full_scaled, y_train_full)

modelo_final_knn = grid_final.best_estimator_
best_params_final = grid_final.best_params_
print(f"Mejores parámetros: {best_params_final}")

x_lags_pred_test = train_d04.values[-12:]
fecha_fin_pred_test = train_d04.index[-1]
tendencia_pred_test = 1.0
mes_pred_test = fecha_fin_pred_test.month
mes_sin_pred_test = np.sin(2 * np.pi * mes_pred_test / 12)
mes_cos_pred_test = np.cos(2 * np.pi * mes_pred_test / 12)
covid_pred_test = 0
x_full_pred_test = np.concatenate([x_lags_pred_test, [tendencia_pred_test, mes_sin_pred_test, mes_cos_pred_test, covid_pred_test]])
X_pred_test = x_full_pred_test.reshape(1, -1)
X_pred_test_scaled = scaler_final.transform(X_pred_test)

pred_test_knn = modelo_final_knn.predict(X_pred_test_scaled).flatten()

real_test = test_d04.values
pred_test = pred_test_knn

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = test_d04.values - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - KNN Regression
--------------------------------------------------
Mejores parámetros: {'estimator__n_neighbors': 10, 'estimator__p': 2, 'estimator__weights': 'uniform'}
RMSE: 310.16
MAE: 290.97
MAPE: 133.58%
SMAPE: 66.69%
MSE: 96197.92


In [50]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_d04.index, y=train_d04.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_d04.index, y=test_d04.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_d04.index, y=pred_test_knn,
    mode='lines+markers', name='Predicción KNN',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'D04 - KNN Regression: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_d04.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_knn,
    mode='lines+markers', name='Predicción KNN',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_d04.values - pred_test_knn

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## Tabla 1 – Métricas Promedio en Validación Cruzada (CV)

| Modelo                     | RMSE    | MAE     | MAPE    | SMAPE   | MSE       |
|----------------------------|---------|---------|---------|---------|-----------|
| Holt‑Winters (SES)         | 285.63  | 237.59  | 45.48%  | 39.96%  | 90992.73  |
| ARIMA(2,0,0)               | 232.13  | 196.18  | 40.09%  | 34.18%  | 55255.02  |
| SARIMA(1,0,0)(0,0,0)[12]   | 270.24  | 225.43  | 37.90%  | 40.82%  | 84260.76  |
| Theta (θ=1.00)             | 290.10  | 244.83  | 49.29%  | 43.56%  | 88579.15  |
| Ridge                      | 268.34  | 231.65  | 43.09%  | 41.00%  | 78102.01  |
| Lasso                      | 294.92  | 265.40  | 46.08%  | 51.05%  | 110410.32 |
| Random Forest              | 272.49  | 241.08  | 45.98%  | 41.97%  | 78496.25  |
| XGBoost                    | 268.48  | 240.61  | 50.21%  | 40.78%  | 74823.33  |
| LightGBM                   | 260.19  | 229.52  | 47.69%  | 39.17%  | 69966.77  |
| KNN                        | 244.22  | 212.02  | 40.46%  | 36.98%  | 61379.50  |

## Tabla 2 – Métricas en Test (2025)

| Modelo                     | RMSE    | MAE     | MAPE    | SMAPE   | MSE       |
|----------------------------|---------|---------|---------|---------|-----------|
| Holt‑Winters (SES)         | 250.61  | 235.12  | 111.22% | 58.52%  | 62805.11  |
| ARIMA(2,0,0)               | 295.74  | 281.03  | 130.28% | 65.49%  | 87463.66  |
| SARIMA(1,0,0)(0,0,0)[12]   | 98.17   | 77.92   | 37.39%  | 26.72%  | 9636.56   |
| Theta (θ=1.00)             | 390.45  | 351.20  | 132.70% | 73.07%  | 152450.62 |
| Ridge                      | 303.12  | 288.11  | 132.22% | 66.56%  | 91880.80  |
| Lasso                      | 314.70  | 294.83  | 133.33% | 67.07%  | 99035.21  |
| Random Forest              | 331.27  | 310.22  | 141.30% | 69.05%  | 109739.14 |
| XGBoost                    | 312.57  | 295.66  | 137.06% | 67.43%  | 97697.70  |
| LightGBM                   | 297.17  | 279.47  | 130.98% | 65.08%  | 88308.78  |
| KNN                        | 310.16  | 290.97  | 133.58% | 66.69%  | 96197.92  |

## Tabla 3 – Errores por Mes en Test (2025)

| Modelo                     | Ene  | Feb  | Mar  | Abr  | May  | Jun  | Jul  | Ago  | Sep  | Oct  | Nov  | Dic  |
|----------------------------|------|------|------|------|------|------|------|------|------|------|------|------|
| Holt‑Winters (SES)         | -199 | -233 | -238 | -283 | -199 | -222 | -205 | -54  | -226 | -233 | -460 | -264 |
| ARIMA(2,0,0)               | -184 | -250 | -274 | -329 | -252 | -279 | -264 | -114 | -287 | -294 | -521 | -325 |
| SARIMA(1,0,0)(0,0,0)[12]   | -102 | -113 | -96  | -120 | -16  | -20  | 14   | 182  | 26   | 34   | -178 | 31   |
| Theta (θ=1.00)             | -270 | -52  | -341 | -527 | -598 | -584 | -325 | 127  | -458 | -468 | -268 | -196 |
| Ridge                      | -250 | -278 | -278 | -358 | -295 | -325 | -286 | -82  | -228 | -280 | -517 | -280 |
| Lasso                      | -280 | -330 | -321 | -418 | -345 | -352 | -293 | -58  | -164 | -253 | -500 | -223 |
| Random Forest              | -220 | -197 | -227 | -397 | -377 | -427 | -334 | -87  | -255 | -325 | -543 | -334 |
| XGBoost                    | -219 | -249 | -245 | -359 | -299 | -337 | -291 | -91  | -280 | -322 | -548 | -309 |
| LightGBM                   | -237 | -327 | -281 | -300 | -236 | -334 | -211 | -74  | -264 | -262 | -536 | -290 |
| KNN                        | -233 | -240 | -354 | -373 | -300 | -346 | -305 | -52  | -204 | -246 | -516 | -324 |

## Interpretación de los Resultados de los Modelos

**a) Desempeño en validación cruzada (CV, 2018‑2024)**  
- Los modelos con menor RMSE promedio en CV fueron ARIMA(2,0,0) (232.1) y KNN (244.2).  
- El SARIMA(1,0,0)(0,0,0)[12] que resultaría ser el mejor a futuro tuvo un RMSE en CV de 270.2, superior al de ARIMA y similar al de otros modelos como LightGBM (260.2) o Ridge (268.3).  
- Esto indica que, durante el período de entrenamiento, las estructuras autorregresivas simples (AR(2)) o basadas en distancia (KNN) lograron un buen ajuste sin grandes diferencias.

**b) Desempeño en el conjunto de prueba (2025)**  
- SARIMA(1,0,0)(0,0,0)[12] destacó de forma contundente, con un RMSE de solo 98.17, MAE de 77.9 y errores porcentuales bajos (MAPE 37.4%, SMAPE 26.7%).  
- Todos los demás modelos fallaron de manera notable, con RMSE entre 250 y 390, MAPE superiores al 110% y errores sistemáticamente negativos (sobreestimación) en casi todos los meses.  
- La tabla de errores mensuales revela que el SARIMA mantuvo equivocaciones pequeñas y con signo variable (ej. enero -102, agosto +182, noviembre -178), mientras que el resto de modelos sobrepronosticaron fuertemente, especialmente en noviembre (errores de hasta -548).  
- El modelo más parecido en CV, ARIMA(2,0,0), tuvo un RMSE de 295.7, casi el triple que SARIMA. La brecha es abrumadora.

---

## ¿Por qué se llegó a estos resultados?

**a) Características de la serie D04 (Paso de semáforo en rojo o señal de pare)**  
- La descomposición STL mostró una tendencia negativa significativa pero débil (pendiente -2.10 comparendos/mes, R²=0.14) y una estacionalidad muy marcada (25.9% de la variabilidad explicada).  
- Los residuos del STL eran ruido blanco, lo que significa que toda la dependencia temporal fue correctamente extraída. Sin embargo, la serie original seguía siendo estacionaria (ADF p<0.001) y la tendencia, aunque presente, no dominaba la dinámica.  
- Esto configura una serie donde el nivel medio desciende lentamente y las fluctuaciones estacionales son intensas pero predecibles, bajo condiciones normales.

**b) Quiebre en 2025**  
- Los modelos que incorporaban explícitamente la estacionalidad histórica (Holt‑Winters, SARIMA con términos estacionales, Ridge/Lasso con variables cíclicas, árboles) extrapolaron un nivel general más alto para 2025, basado en los patrones de años anteriores.  
- Sin embargo, en 2025 la serie mostró valores persistentemente más bajos de lo esperado, especialmente en la segunda mitad del año (noviembre con un error masivo). Esto sugiere que la caída se aceleró o que la estructura estacional se alteró, posiblemente por cambios en la fiscalización o en el comportamiento de los conductores.  
- SARIMA(1,0,0)(0,0,0)[12], al ser un simple AR(1) sin componentes estacionales, no impuso ningún patrón cíclico fijo. Sus predicciones se anclan fuertemente en el último valor observado y en la media reciente, por lo que se adaptó automáticamente al nuevo nivel más bajo de la serie.  
- Al no intentar modelar una tendencia lineal fija, tampoco extrapoló un declive excesivo; simplemente proyectó un nivel cercano al último dato, lo que resultó en errores mucho menores que los modelos que “esperaban” una recuperación estacional o un nivel medio histórico más elevado.  
- Los modelos restantes, al incluir estacionalidad y/o una tendencia lineal, sobrepronosticaron de manera consistente (errores negativos), porque en 2025 la realidad fue más baja y con un ciclo atenuado.

**c) Lección metodológica**  
- Este es un caso clásico donde la simplicidad supera a la complejidad ante un cambio de régimen no anticipado. Los residuos ruido blanco del STL validaban el ajuste histórico, pero no garantizaban la estabilidad futura. La robustez del AR(1) frente a quiebres estacionales lo convirtió en la mejor opción para este año atípico.

---

## Modelo Ideal para Predecir 2026

Se recomienda utilizar el modelo SARIMA(1,0,0)(0,0,0)[12] para el pronóstico de 2026 del código D04.

**Justificación:**  
- Fue el único modelo que ofreció un desempeño aceptable en el año de prueba (2025), con un RMSE de 98, frente a valores superiores a 250 del resto. La magnitud de la diferencia excluye cualquier otra opción.  
- Como modelo autorregresivo puro, sus predicciones se basan principalmente en la inercia reciente de la serie, sin asumir que los patrones estacionales del pasado se repetirán con la misma intensidad. Si la caída observada en 2025 persiste, el AR(1) continuará siguiéndola; si la serie se estabiliza, convergerá suavemente a la nueva media.  
- Es un modelo eficiente (un solo parámetro autorregresivo más una constante) que puede reestimarse fácilmente conforme se incorporen nuevos datos, manteniendo su vigencia.

## Pronóstico de la Infracción D04 para el Año 2026

Una vez seleccionado el mejor modelo predictivo (SARIMA con orden autorregresivo de primer orden (1,0,0) y sin componente estacional estacional, es decir, (0,0,0,12)), se procede a realizar el pronóstico del volumen de infracciones por paso de semáforo en rojo para los 12 meses del año 2026. El modelo se entrena con la serie completa de 96 meses (enero 2018 a diciembre 2025) y genera predicciones puntuales con sus respectivos intervalos de confianza para cada mes de 2026. Los resultados se visualizan mediante dos gráficos interactivos: el primero muestra la serie histórica completa junto con la proyección hacia 2026, con una línea vertical discontinua que marca el inicio del período pronosticado; el segundo presenta exclusivamente el pronóstico mensual para 2026, facilitando la interpretación de la tendencia esperada mes a mes. Esta visualización permite anticipar el comportamiento futuro de la infracción y apoyar la toma de decisiones en materia de control de semáforos y seguridad vial.

In [51]:
mejor_order = (1, 0, 0)
mejor_seasonal = (0, 0, 0, 12)

modelo_sarima_2026 = SARIMAX(
    serie_d04,
    order=mejor_order,
    seasonal_order=mejor_seasonal,
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

forecast_2026_d04 = modelo_sarima_2026.forecast(steps=12)

idx_forecast = pd.date_range(start='2026-01-01', periods=12, freq='MS')
forecast_2026_d04.index = idx_forecast

fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=serie_d04.index,
    y=serie_d04.values,
    mode='lines+markers',
    name='Histórico (2018-2025)',
    line=dict(color='cornflowerblue', width=2),
    marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=forecast_2026_d04.index,
    y=forecast_2026_d04.values,
    mode='lines+markers',
    name='Predicción SARIMA(1,0,0)(0,0,0)[12]',
    line=dict(color='red', width=2),
    marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(
    x='2026-01-01',
    line_width=2,
    line_dash='dash',
    line_color='darkgray',
    opacity=1
)

fig1.add_annotation(
    x='2026-01-01',
    y=1,
    yref='paper',
    text='<b>Inicio 2026</b>',
    showarrow=False,
    font=dict(size=12, color='darkgray'),
    xanchor='left',
    yanchor='bottom'
)

fig1.update_layout(
    title=dict(
        text='D04 - SARIMA(1,0,0)(0,0,0)[12]: Predicción 2026<br>'
             '<sup>Modelo entrenado con datos hasta diciembre 2025</sup>',
        x=0.5,
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white',
    hovermode='x unified',
    legend=dict(
        x=0.01,
        y=0.99,
        bgcolor='rgba(255,255,255,0.7)',
        bordercolor='lightgray',
        borderwidth=1
    ),
    width=1055,
    height=500
)

fig1.show()

meses_2026 = ['Enero','Febrero','Marzo','Abril','Mayo','Junio',
              'Julio','Agosto','Septiembre','Octubre','Noviembre','Diciembre']

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses_2026,
    y=forecast_2026_d04.values,
    mode='lines+markers',
    name='Predicción SARIMA(1,0,0)(0,0,0)[12]',
    line=dict(color='red', width=2),
    marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(
        text='Pronóstico D04 para el año 2026',
        x=0.5,
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white',
    hovermode='x unified',
    showlegend=False,
    width=1055,
    height=500
)

fig2.show()